# NB_SILVER_2 — Sommaire

> **Pipeline NovaRetail — Couche Silver**  
> Transformation Bronze → Silver (nettoyage, typage, normalisation, contrôles qualité)  
> Microsoft Fabric Lakehouse · PySpark

---

## Initialisation

- [En-tête — Contexte et tables produites](#entete)
- [Cellule 1 — Imports & Configuration](#cellule-1)
- [Cellule 2 — Schémas Delta Silver](#cellule-2)
- [Cellule 3 — Fonctions utilitaires de nettoyage (UDFs)](#cellule-3)

---

## Chargement & contrôle qualité Bronze

- [Cellule 4 — Chargement Bronze & contrôles qualité étendus](#cellule-4)

---

## Construction des référentiels Silver

- [Cellule 5 — `silver_clients` — Extraction et normalisation des clients](#cellule-5)
- [Cellule 6 — `silver_commerciaux` — Extraction et normalisation des commerciaux](#cellule-6)
- [Cellule 7 — `silver_articles` — Extraction et normalisation des articles](#cellule-7)

---

## Construction des tables transactionnelles Silver

- [Cellule 8 — `silver_commandes` — Commandes typées et validées](#cellule-8)
- [Cellule 9 — `silver_lignes_commande` — Lignes articles typées](#cellule-9)

---

## Contrôles qualité & clôture

- [Cellule 10 — Audit Silver](#cellule-10)
- [Cellule 11 — Vérifications post-Silver et analyses exploratoires](#cellule-11)
- [Cellule 12 — Diagnostic des anomalies de montant](#cellule-12)
- [Cellule 13 — Contrôles de cohérence inter-tables Silver](#cellule-13)
- [Cellule 14 — Rapport final, score qualité global et nettoyage](#cellule-14)


<a id="entete"></a>

## En-tête du notebook — Contexte et tables produites

Ce bloc de commentaire décrit l'objectif global du notebook Silver (NB_02) dans le pipeline **NovaRetail**.

Il s'inscrit dans une architecture **Medallion** (Bronze → Silver → Gold) hébergée sur **Microsoft Fabric Lakehouse** avec **PySpark** et le format **Delta**.

À l'issue de ce notebook, cinq tables Silver sont produites :
- `silver_clients` — référentiel clients dédupliqué
- `silver_commerciaux` — référentiel commerciaux dédupliqué
- `silver_articles` — référentiel articles dédupliqué
- `silver_commandes` — commandes nettoyées, typées et validées
- `silver_lignes_commande` — lignes articles nettoyées et typées

In [1]:
# =============================================================================
# NOVARETAIL — NOTEBOOK SILVER (NB_02)
# Transformation Bronze → Silver (nettoyage, typage, normalisation)
# Microsoft Fabric Lakehouse — PySpark
#
# Auteur      : NovaRetail Data Team
# Version     : 1.0
# Couche      : Silver (Cleaned & Normalized)
#
# Tables produites :
#   silver_clients           — référentiel clients dédupliqué
#   silver_commerciaux       — référentiel commerciaux dédupliqué
#   silver_articles          — référentiel articles dédupliqué
#   silver_commandes         — commandes nettoyées et typées
#   silver_lignes_commande   — lignes articles nettoyées et typées
# =============================================================================

StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 3, Finished, Available, Finished, False)

<a id="cellule-1"></a>

## Cellule 1 — Imports & Configuration

Initialisation de l'environnement d'exécution PySpark et définition des chemins ABFSS vers le Lakehouse.

**Points clés :**
- `SparkSession.builder.getOrCreate()` récupère la session Spark existante (fournie par Fabric), sans en créer une nouvelle.
- Les deux options `spark.conf.set` corrigent des problèmes de compatibilité de sérialisation des dates Parquet (`CORRECTED`) et d'interprétation du format de dates legacy (`LEGACY`).
- Les chemins `LAKEHOUSE_ROOT` utilisent le protocole **ABFSS** (Azure Blob File System Secure), requis pour accéder à OneLake depuis Fabric.
- `SILVER_RUN_ID` est un identifiant unique horodaté qui tracera chaque exécution dans la table d'audit.

In [2]:
# CELLULE 1 — Imports & Configuration
# -----------------------------------------------------------------------------

import re
import hashlib
from datetime import datetime, timezone

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, LongType,
    IntegerType, DecimalType, DateType, BooleanType
)

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

WORKSPACE_NAME = "Projet_technique"
LAKEHOUSE_NAME = "Bons_de_commandes"

LAKEHOUSE_ROOT = (
    f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_NAME}.Lakehouse"
)

# ── Tables Bronze (sources) ───────────────────────────────────────────────────
BRONZE_HEADERS = f"{LAKEHOUSE_ROOT}/Tables/bronze_commandes_headers"
BRONZE_LIGNES  = f"{LAKEHOUSE_ROOT}/Tables/bronze_commandes_lignes"

# ── Tables Silver (destinations) ─────────────────────────────────────────────
SILVER_CLIENTS          = f"{LAKEHOUSE_ROOT}/Tables/silver_clients"
SILVER_COMMERCIAUX      = f"{LAKEHOUSE_ROOT}/Tables/silver_commerciaux"
SILVER_ARTICLES         = f"{LAKEHOUSE_ROOT}/Tables/silver_articles"
SILVER_COMMANDES        = f"{LAKEHOUSE_ROOT}/Tables/silver_commandes"
SILVER_LIGNES_COMMANDE  = f"{LAKEHOUSE_ROOT}/Tables/silver_lignes_commande"
SILVER_AUDIT            = f"{LAKEHOUSE_ROOT}/Tables/silver_audit"

SILVER_TIMESTAMP = datetime.now(timezone.utc)
SILVER_RUN_ID    = f"SILVER_{SILVER_TIMESTAMP.strftime('%Y%m%d_%H%M%S')}"

print(f"[CONFIG] Workspace  : {WORKSPACE_NAME}")
print(f"[CONFIG] Lakehouse  : {LAKEHOUSE_NAME}")
print(f"[CONFIG] Timestamp  : {SILVER_TIMESTAMP.isoformat()}")
print(f"[CONFIG] Run ID     : {SILVER_RUN_ID}")

StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 4, Finished, Available, Finished, False)

[CONFIG] Workspace  : Projet_technique
[CONFIG] Lakehouse  : Bons_de_commandes
[CONFIG] Timestamp  : 2026-06-08T12:43:54.095782+00:00
[CONFIG] Run ID     : SILVER_20260608_124354


<a id="cellule-2"></a>

## Cellule 2 — Schémas Delta Silver

Déclaration explicite des schémas **StructType** de toutes les tables Silver et de la table d'audit.

**Pourquoi déclarer les schémas manuellement ?**
- Évite l'inférence automatique de types par Spark, source d'erreurs silencieuses (ex. un montant inféré en `string` au lieu de `DecimalType`).
- Garantit la stabilité du schéma entre les runs et la compatibilité avec les outils BI downstream (Power BI, Semantic Layer).
- Les champs `nullable=False` sur les clés primaires synthétiques (`silver_*_id`) et les timestamps permettent d'identifier toute anomalie de génération d'ID dès l'écriture Delta.

**Choix de typage notables :**
- `DecimalType(12,2)` pour les montants : précision comptable, pas de dérive floating-point.
- `DateType()` pour les dates (pas `TimestampType`) : les dates métier (commande, livraison) n'ont pas de composante heure utile.
- `StringType()` pour le SIREN, le code postal et le numéro de commande : ces champs sont des identifiants à traiter comme chaînes (zéros de tête, validation métier séparée).


In [3]:
# CELLULE 2 — Schémas Delta Silver
# -----------------------------------------------------------------------------

SCHEMA_SILVER_CLIENTS = StructType([
    StructField("silver_client_id",   StringType(),   False), # PK SHA256(siren ou nom_client)
    StructField("nom_client",         StringType(),   True),
    StructField("adresse_client",     StringType(),   True),
    StructField("ville_client",       StringType(),   True),
    StructField("code_postal",        StringType(),   True),
    StructField("siren",              StringType(),   True),
    StructField("contact_nom",        StringType(),   True),
    StructField("contact_email",      StringType(),   True),
    StructField("contact_telephone",  StringType(),   True),
    StructField("bronze_id_source",   StringType(),   True),  # bronze_id du 1er BDC source
    StructField("nb_commandes",       IntegerType(),  True),  # nombre de BDC connus
    StructField("created_at",         TimestampType(),False),
    StructField("updated_at",         TimestampType(),False),
])

SCHEMA_SILVER_COMMERCIAUX = StructType([
    StructField("silver_commercial_id",  StringType(),   False),
    StructField("nom_commercial",        StringType(),   False),
    StructField("fonction_commercial",   StringType(),   True),
    StructField("email",                 StringType(),   True), 
    StructField("nb_commandes",          IntegerType(),  True),
    StructField("created_at",            TimestampType(),False),
    StructField("updated_at",            TimestampType(),False),
])

SCHEMA_SILVER_ARTICLES = StructType([
    StructField("silver_article_id",  StringType(),   False),
    StructField("reference_article",  StringType(),   True),
    StructField("designation",        StringType(),   True),
    StructField("calibre",            StringType(),   True),
    StructField("categorie",          StringType(),   True),  # FRUIT / LEGUME / INCONNU
    StructField("origine_pays",       StringType(),   True),  # France / Espagne / ...
    StructField("unite_vente",        StringType(),   True),
    StructField("created_at",         TimestampType(),False),
    StructField("updated_at",         TimestampType(),False),
])

SCHEMA_SILVER_COMMANDES = StructType([
    StructField("silver_commande_id",         StringType(),    False),
    StructField("bronze_id",                  StringType(),    False),
    StructField("silver_client_id",           StringType(),    True),
    StructField("silver_commercial_id",       StringType(),    True),
    StructField("numero_commande",            StringType(),    True),
    StructField("date_commande",              DateType(),      True),
    StructField("date_livraison_souhaitee",   DateType(),      True),
    StructField("heure_livraison_souhaitee",  StringType(),    True),
    StructField("lieu_livraison",             StringType(),    True),
    StructField("mode_livraison",             StringType(),    True),
    StructField("mode_reglement",             StringType(),    True),  
    StructField("incoterm",                   StringType(),    True), 
    StructField("ref_contrat",                StringType(),    True),
    StructField("total_ht",                   DecimalType(12,2),True),
    StructField("total_ttc",                  DecimalType(12,2),True),
    StructField("tva_taux",                   DecimalType(5,2), True),
    StructField("devise",                     StringType(),    True),
    StructField("fournisseur",                StringType(),    True),
    StructField("nb_lignes",                  IntegerType(),   True),
    StructField("statut_validation",          StringType(),    False), 
    StructField("motif_rejet",                StringType(),    True),
    StructField("periode_zip",                StringType(),    True),
    StructField("format_source",              StringType(),    True),
    StructField("source_fichier",             StringType(),    True),
    StructField("ingestion_timestamp",        TimestampType(), True),
    StructField("silver_timestamp",           TimestampType(), False),
])

SCHEMA_SILVER_LIGNES = StructType([
    StructField("silver_ligne_id",       StringType(),    False),
    StructField("silver_commande_id",    StringType(),    False),
    StructField("silver_article_id",     StringType(),    True),
    StructField("bronze_ligne_id",       StringType(),    True),
    StructField("index_ligne",           IntegerType(),   True),
    StructField("reference_article",     StringType(),    True),
    StructField("designation",           StringType(),    True),
    StructField("calibre",               StringType(),    True),
    StructField("quantite",              DecimalType(10,3),True),
    StructField("unite",                 StringType(),    True),
    StructField("prix_unitaire_ht",      DecimalType(10,4),True),
    StructField("remise_pct",            DecimalType(5,2), True),
    StructField("montant_ht_ligne",      DecimalType(12,2),True),
    StructField("montant_ht_calcule",    DecimalType(12,2),True), # quantite × prix_unitaire
    StructField("ecart_montant",         DecimalType(12,2),True), # montant_ht_ligne - montant_ht_calcule
    StructField("tva_taux",              DecimalType(5,2), True),
    StructField("date_livraison_ligne",  DateType(),      True),
    StructField("statut_ligne",          StringType(),    False), # OK / ANOMALIE_MONTANT / INCOMPLET
    StructField("silver_timestamp",      TimestampType(), False),
])

SCHEMA_SILVER_AUDIT = StructType([
    StructField("run_id",                    StringType(),   False),
    StructField("silver_timestamp",          TimestampType(),False),
    StructField("nb_headers_bronze",         IntegerType(),  True),
    StructField("nb_lignes_bronze",          IntegerType(),  True),
    StructField("nb_clients_crees",          IntegerType(),  True),
    StructField("nb_commerciaux_crees",      IntegerType(),  True),
    StructField("nb_articles_crees",         IntegerType(),  True),
    StructField("nb_commandes_ok",           IntegerType(),  True),
    StructField("nb_commandes_incomplet",    IntegerType(),  True),
    StructField("nb_commandes_doublon",      IntegerType(),  True),
    StructField("nb_lignes_ok",              IntegerType(),  True),
    StructField("nb_lignes_anomalie",        IntegerType(),  True),
    StructField("duree_secondes",            IntegerType(),  True),
])

print("[SCHÉMAS] Définis : clients, commerciaux, articles, commandes, lignes, audit")

StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 5, Finished, Available, Finished, False)

[SCHÉMAS] Définis : clients, commerciaux, articles, commandes, lignes, audit


<a id="cellule-3"></a>

## Cellule 3 — Fonctions utilitaires de nettoyage (UDFs)

Définition des fonctions Python de nettoyage, puis enregistrement en tant qu'**UDFs Spark** pour application distribuée sur les DataFrames.

**Fonctions définies :**

| UDF | Rôle |
|-----|------|
| `nettoyer_montant` | Normalise les montants hétérogènes (`'7 968,78 €'`, `'94.50'`) en `float` |
| `nettoyer_date` | Parse les dates multi-format (`DD/MM/YYYY`, `YYYY-MM-DD`, `DD/MM/YY`) en ISO `yyyy-MM-dd` |
| `extraire_categorie` | Catégorise un article (`FRUIT` / `LEGUME` / `INCONNU`) par règles sur la référence puis la désignation |
| `extraire_origine` | Extrait le pays d'origine depuis la désignation (pattern `– Pays` en fin de chaîne) |
| `extraire_calibre` | Extrait la mention de calibre si présente (`Cal. 5`, `Cat. I`, `Extra`) |
| `generer_id` | Génère un identifiant SHA256 déterministe depuis une ou plusieurs clés métier |

**Pourquoi des UDFs Python plutôt que des fonctions Spark natives ?**  
Les logiques de parsing (regex multi-pattern, Luhn) sont trop complexes pour être exprimées en pur DSL Spark. L'alternative serait des UDFs Scala/Java (plus performantes), mais la maintenabilité Python prime ici. Pour des volumes très élevés, migrer vers `pandas_udf` (vectorisé Arrow) serait la prochaine optimisation.


In [4]:
# CELLULE 3 — Fonctions utilitaires de nettoyage
# -----------------------------------------------------------------------------

# ── UDFs de nettoyage ─────────────────────────────────────────────────────────

def nettoyer_montant(val):
    """
    Convertit une chaîne montant en float.
    Gère : '7 968.78 €', '94,50', '1 397,80', '0,63'
    """
    if val is None:
        return None
    val = str(val).strip()
    val = re.sub(r'[€$£EUR\s]', '', val)   # supprimer symboles et espaces
    val = val.replace(',', '.')             # virgule → point
    val = re.sub(r'\.(?=.*\.)', '', val)   # supprimer tous les points sauf le dernier
    try:
        return float(val)
    except (ValueError, TypeError):
        return None

def nettoyer_date(val):
    """
    Convertit une chaîne date en string ISO (yyyy-MM-dd).
    Gère : '03/06/2024', '03-06-2024', '03.06.2024'
    """
    if val is None:
        return None
    val = str(val).strip()
    patterns = [
        (r'^(\d{2})[/\-\.](\d{2})[/\-\.](\d{4})$', '{2}-{1}-{0}'),  # DD/MM/YYYY
        (r'^(\d{4})[/\-\.](\d{2})[/\-\.](\d{2})$', '{0}-{1}-{2}'),  # YYYY-MM-DD
        (r'^(\d{2})[/\-\.](\d{2})[/\-\.](\d{2})$', '20{2}-{1}-{0}'), # DD/MM/YY
    ]
    for pattern, fmt in patterns:
        m = re.match(pattern, val)
        if m:
            parts = list(m.groups())
            return fmt.format(*parts)
    return None

def extraire_categorie(reference, designation):
    """
    Catégorise un article depuis sa référence ou sa désignation.
    F-xxx → FRUIT, L-xxx → LEGUME, sinon INCONNU
    """
    if reference and str(reference).upper().startswith('F-'):
        return 'FRUIT'
    if reference and str(reference).upper().startswith('L-'):
        return 'LEGUME'
    if designation:
        des = str(designation).lower()
        fruits = ['pomme', 'poire', 'orange', 'citron', 'fraise', 'framboise',
                  'cerise', 'abricot', 'pêche', 'melon', 'pastèque', 'raisin',
                  'myrtille', 'banane', 'ananas', 'kiwi', 'mangue', 'figue']
        legumes = ['tomate', 'carotte', 'poivron', 'courgette', 'aubergine',
                   'brocoli', 'chou', 'salade', 'laitue', 'épinard', 'poireau',
                   'oignon', 'ail', 'champignon', 'haricot', 'petit pois',
                   'roquette', 'radis', 'concombre', 'pomme de terre']
        for f in fruits:
            if f in des:
                return 'FRUIT'
        for l in legumes:
            if l in des:
                return 'LEGUME'
    return 'INCONNU'

def extraire_origine(designation):
    """
    Extrait le pays d'origine depuis la désignation.
    Ex : 'Pommes de terre Agria – France' → 'France'
         'Oranges Valencia – Espagne' → 'Espagne'
    """
    if designation is None:
        return None
    m = re.search(r'[–\-]\s*([A-ZÀ-Ÿa-zà-ÿ\s]+)$', str(designation).strip())
    if m:
        pays = m.group(1).strip()
        if len(pays) > 2 and len(pays) < 40:
            return pays
    return None

def extraire_calibre(designation):
    """
    Extrait le calibre depuis la désignation si présent.
    Ex : 'Oranges Valencia – Espagne Cal. 5' → 'Cal. 5'
    """
    if designation is None:
        return None
    m = re.search(r'(Cal\.\s*[\w\+\-]+|Cat\.\s*[IVX]+|Grade\s*\w+|Extra)', str(designation))
    if m:
        return m.group(1).strip()
    return None

def generer_id(*parties):
    """Génère un ID SHA256 depuis plusieurs parties."""
    import hashlib
    cle = "::".join(str(p) for p in parties if p)
    return hashlib.sha256(cle.encode("utf-8")).hexdigest()

# Enregistrement des UDFs Spark
udf_montant    = F.udf(nettoyer_montant,  "double")
udf_date       = F.udf(nettoyer_date,     StringType())
udf_categorie  = F.udf(extraire_categorie, StringType())
udf_origine    = F.udf(extraire_origine,  StringType())
udf_calibre    = F.udf(extraire_calibre,  StringType())
udf_id         = F.udf(generer_id,        StringType())

print("[UTILS] UDFs enregistrées : montant, date, categorie, origine, calibre, id")

StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 6, Finished, Available, Finished, False)

[UTILS] UDFs enregistrées : montant, date, categorie, origine, calibre, id


<a id="cellule-4"></a>

## Cellule 4 — Chargement des données Bronze et contrôles qualité étendus

Chargement des deux tables Bronze sources, puis exécution d'une série de **contrôles qualité préventifs** avant toute transformation Silver.

**Étapes réalisées :**

1. **Lecture Delta filtrée** : seuls les headers avec `statut_parsing = 'SUCCESS'` sont chargés — les fichiers ayant échoué en Bronze ne doivent pas polluer la couche Silver.

2. **Taux de remplissage** des champs clés : pour chaque champ critique (numéro commande, montants, SIREN, email…), calcul du pourcentage de valeurs non-nulles avec seuils visuels (✅ > 90%, ⚠️ > 50%, ❌ sinon).

3. **Répartition par format source** : agrégation par `format_source` pour détecter des disparités de qualité selon l'origine des fichiers (PDF, DOCX, Excel…).

4. **Détection de doublons Bronze** : via une fenêtre `ROW_NUMBER()` partitionnée sur `numero_commande_raw`, les doublons sont comptabilisés avant dédoublonnage Silver.

5. **Idempotence** : lecture des `bronze_id` déjà présents en Silver pour filtrer les headers et lignes déjà traités. Garantit que chaque run est **safe-to-replay** sans duplication.


In [5]:
# CELLULE 4 — Chargement des données Bronze + contrôles qualité étendus
# -----------------------------------------------------------------------------

print("\n[BRONZE] Chargement des tables Bronze...")

df_bronze_headers = (spark.read.format("delta").load(BRONZE_HEADERS)
    .filter(F.col("statut_parsing") == "SUCCESS")
)
df_bronze_lignes = spark.read.format("delta").load(BRONZE_LIGNES)

nb_headers = df_bronze_headers.count()
nb_lignes  = df_bronze_lignes.count()

print(f"  Headers Bronze : {nb_headers:,} enregistrement(s)")
print(f"  Lignes Bronze  : {nb_lignes:,} enregistrement(s)")

# ── Qualité Bronze : taux de remplissage champs clés ─────────────────────────
print("\n[QUALITE BRONZE] Taux de remplissage champs clés :")
CHAMPS_QUALITE = [
    "numero_commande_raw", "date_commande_raw", "total_ht_raw",
    "total_ttc_raw", "tva_raw", "commercial_nom_raw", "lieu_livraison_raw",
    "client_nom_raw", "client_siren_raw", "client_contact_email_raw",
    "client_contact_tel_raw", "mode_reglement_raw", "incoterm_raw",
    "fournisseur_email_raw",
]
total = nb_headers
for col_name in CHAMPS_QUALITE:
    try:
        nb_nn = df_bronze_headers.filter(F.col(col_name).isNotNull()).count()
        pct   = round(nb_nn / total * 100, 1) if total > 0 else 0
        ico   = "✅" if pct > 90 else ("⚠️" if pct > 50 else "❌")
        print(f"  {ico} {col_name:35s} : {pct:5.1f}%")
    except Exception:
        print(f"  ⬜ {col_name:35s} : champ absent du schéma Bronze")

# ── Qualité Bronze par format source ─────────────────────────────────────────
print("\n[QUALITE BRONZE] Répartition par format source :")
df_bronze_headers.groupBy("format_source").agg(
    F.count("bronze_id").alias("nb_fichiers"),
    F.round(F.avg(F.col("nb_lignes_articles").cast("double")), 1).alias("moy_lignes"),
    F.sum(F.when(F.col("numero_commande_raw").isNotNull(), 1).otherwise(0)).alias("avec_numero"),
    F.sum(F.when(F.col("total_ht_raw").isNotNull(), 1).otherwise(0)).alias("avec_ht"),
).orderBy("format_source").show(20)

# ── Doublons potentiels en Bronze ─────────────────────────────────────────────
print("[QUALITE BRONZE] Détection doublons numéro commande :")
_w_dup = Window.partitionBy("numero_commande_raw").orderBy(F.desc("ingestion_timestamp"))
df_dup_bronze = (df_bronze_headers
    .filter(F.col("numero_commande_raw").isNotNull())
    .withColumn("_rn", F.row_number().over(_w_dup))
    .filter(F.col("_rn") > 1)
)
nb_dup_bronze = df_dup_bronze.count()
print(f"  {'⚠️' if nb_dup_bronze > 0 else '✅'} Doublons numéro commande en Bronze : {nb_dup_bronze:,}")

# ── Idempotence Silver ────────────────────────────────────────────────────────
def charger_silver_bronze_ids() -> set:
    try:
        df  = spark.read.format("delta").load(SILVER_COMMANDES)
        ids = {r["bronze_id"] for r in df.select("bronze_id").collect()}
        print(f"[IDEMPOTENCE] {len(ids)} commande(s) déjà en Silver")
        return ids
    except Exception:
        print("[IDEMPOTENCE] Table Silver inexistante — première ingestion")
        return set()

BRONZE_IDS_SILVER = charger_silver_bronze_ids()

if BRONZE_IDS_SILVER:
    df_bronze_headers_new = df_bronze_headers.filter(
        ~F.col("bronze_id").isin(list(BRONZE_IDS_SILVER))
    )
    nb_new = df_bronze_headers_new.count()
    print(f"[IDEMPOTENCE] {nb_new:,} nouveau(x) header(s) à traiter "
          f"({nb_headers - nb_new:,} déjà en Silver)")
else:
    df_bronze_headers_new = df_bronze_headers
    nb_new = nb_headers

bronze_ids_new = {r["bronze_id"] for r in df_bronze_headers_new.select("bronze_id").collect()}
if bronze_ids_new:
    df_bronze_lignes_new = df_bronze_lignes.filter(
        F.col("bronze_id").isin(list(bronze_ids_new))
    )
else:
    df_bronze_lignes_new = df_bronze_lignes

print(f"[IDEMPOTENCE] {df_bronze_lignes_new.count():,} ligne(s) bronze à traiter")


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 7, Finished, Available, Finished, False)


[BRONZE] Chargement des tables Bronze...
  Headers Bronze : 62,539 enregistrement(s)
  Lignes Bronze  : 484,066 enregistrement(s)

[QUALITE BRONZE] Taux de remplissage champs clés :
  ✅ numero_commande_raw                 : 100.0%
  ✅ date_commande_raw                   :  99.4%
  ✅ total_ht_raw                        : 100.0%
  ✅ total_ttc_raw                       : 100.0%
  ⚠️ tva_raw                             :  75.2%
  ⚠️ commercial_nom_raw                  :  63.7%
  ⚠️ lieu_livraison_raw                  :  75.4%
  ❌ client_nom_raw                      :  19.1%
  ❌ client_siren_raw                    :  19.1%
  ❌ client_contact_email_raw            :  19.1%
  ❌ client_contact_tel_raw              :  19.1%
  ❌ mode_reglement_raw                  :  19.1%
  ❌ incoterm_raw                        :  19.1%
  ✅ fournisseur_email_raw               : 100.0%

[QUALITE BRONZE] Répartition par format source :
+-------------+-----------+----------+-----------+-------+
|format_source|nb_f

<a id="cellule-5"></a>

## Cellule 5 — Extraction et normalisation des clients (`silver_clients`)

Construction du **référentiel clients dédupliqué** depuis les headers Bronze.

**Logique de déduplication :**
- La clé de déduplication (`cle_dedup`) est dérivée du champ `lieu_livraison_raw` normalisé (minuscules, espaces consolidés).
- L'ID client (`silver_client_id`) est un SHA256 de cette clé — déterministe et stable entre les runs, sans séquence auto-incrémentée.
- Un `ROW_NUMBER()` sur fenêtre triée par `ingestion_timestamp` conserve le **premier enregistrement connu** comme référence maître.

**Enrichissements :**
- Extraction de la ville et du code postal depuis `lieu_livraison_raw` par regex.
- Comptage du nombre de commandes associées par client (`nb_commandes`) via une agrégation préalable.

**Contrôles qualité post-écriture :**
- Champs obligatoires manquants (nom, SIREN, email, code postal, ville).
- Validation de format : email (regex RFC-5322 simplifié), code postal (5 chiffres), SIREN (9 chiffres + algorithme de Luhn).
- Détection de SIREN dupliqués sur des `silver_client_id` distincts (fusion client potentielle à investiguer).


In [6]:
# CELLULE 5 — Extraction et normalisation des CLIENTS
# -----------------------------------------------------------------------------

print("\n[SILVER] Construction silver_clients...")

def extraire_nom_client(lieu):
    if lieu is None:
        return None
    return str(lieu).strip()[:200]

def extraire_ville(lieu):
    if lieu is None:
        return None
    m = re.search(r'\d{5}\s+([A-ZÀ-Ÿa-zà-ÿ\s\-]+)$', str(lieu))
    if m:
        return m.group(1).strip()
    return None

def extraire_cp(lieu):
    if lieu is None:
        return None
    m = re.search(r'(\d{5})', str(lieu))
    if m:
        return m.group(1)
    return None

def valider_email(email):
    """Valide le format d'un email."""
    if email is None:
        return False
    return bool(re.match(r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$', str(email).strip()))

def valider_cp(cp):
    """Valide un code postal français (5 chiffres)."""
    if cp is None:
        return False
    return bool(re.match(r'^\d{5}$', str(cp).strip()))

def valider_siren(siren):
    """Valide un SIREN (9 chiffres, algorithme de Luhn)."""
    if siren is None:
        return False
    s = re.sub(r'\s', '', str(siren))
    if not re.match(r'^\d{9}$', s):
        return False
    # Luhn check
    total = 0
    for i, digit in enumerate(reversed(s)):
        n = int(digit)
        if i % 2 == 1:
            n *= 2
            if n > 9:
                n -= 9
        total += n
    return total % 10 == 0

udf_nom_client  = F.udf(extraire_nom_client, StringType())
udf_ville       = F.udf(extraire_ville,      StringType())
udf_cp          = F.udf(extraire_cp,         StringType())
udf_valid_email = F.udf(valider_email,       BooleanType())
udf_valid_cp    = F.udf(valider_cp,          BooleanType())
udf_valid_siren = F.udf(valider_siren,       BooleanType())

df_clients_raw = (df_bronze_headers_new
    .select(
        F.col("bronze_id"),
        F.col("lieu_livraison_raw"),
        F.col("client_nom_raw"),
        F.col("client_siren_raw"),
        F.col("client_contact_nom_raw"),
        F.col("client_contact_email_raw"),
        F.col("client_contact_tel_raw"),
        F.col("numero_commande_raw"),
        F.col("ingestion_timestamp"),
    )
    .withColumn("adresse_client",  udf_nom_client(F.col("lieu_livraison_raw")))
    .withColumn("ville_client",    udf_ville(F.col("lieu_livraison_raw")))
    .withColumn("code_postal",     udf_cp(F.col("lieu_livraison_raw")))
    .withColumn("cle_dedup",
        F.lower(F.regexp_replace(F.coalesce(F.col("lieu_livraison_raw"), F.lit("")), r'\s+', ' ')))
    .withColumn("silver_client_id", udf_id(F.col("cle_dedup")))
)

df_nb_commandes_client = (df_clients_raw
    .groupBy("silver_client_id")
    .agg(F.count("bronze_id").alias("nb_commandes"))
)

w_client = Window.partitionBy("silver_client_id").orderBy("ingestion_timestamp")
df_clients_dedup = (df_clients_raw
    .withColumn("rn", F.row_number().over(w_client))
    .filter(F.col("rn") == 1)
    .join(df_nb_commandes_client, on="silver_client_id", how="left")
    .select(
        F.col("silver_client_id"),
        F.trim(F.col("client_nom_raw")).alias("nom_client"),
        F.col("adresse_client"),
        F.col("ville_client"),
        F.col("code_postal"),
        F.trim(F.col("client_siren_raw")).alias("siren"),
        F.trim(F.col("client_contact_nom_raw")).alias("contact_nom"),
        F.trim(F.col("client_contact_email_raw")).alias("contact_email"),
        F.trim(F.col("client_contact_tel_raw")).alias("contact_telephone"),
        F.col("bronze_id").alias("bronze_id_source"),
        F.col("nb_commandes"),
        F.lit(SILVER_TIMESTAMP).alias("created_at"),
        F.lit(SILVER_TIMESTAMP).alias("updated_at"),
    )
)

nb_clients = df_clients_dedup.count()
print(f"  Clients dédupliqués : {nb_clients:,}")

(df_clients_dedup.write
    .format("delta").mode("overwrite")
    .option("mergeSchema", "true")
    .save(SILVER_CLIENTS))

# ── Contrôles qualité clients ──────────────────────────────────────────────────
print("\n  [QUALITÉ CLIENTS]")
nb_sans_nom   = df_clients_dedup.filter(F.col("nom_client").isNull()).count()
nb_sans_siren = df_clients_dedup.filter(F.col("siren").isNull()).count()
nb_sans_email = df_clients_dedup.filter(F.col("contact_email").isNull()).count()
nb_sans_cp    = df_clients_dedup.filter(F.col("code_postal").isNull()).count()
nb_sans_ville = df_clients_dedup.filter(F.col("ville_client").isNull()).count()

# Validations format
nb_email_invalide = df_clients_dedup.filter(
    F.col("contact_email").isNotNull() & ~udf_valid_email(F.col("contact_email"))
).count()
nb_cp_invalide = df_clients_dedup.filter(
    F.col("code_postal").isNotNull() & ~udf_valid_cp(F.col("code_postal"))
).count()
nb_siren_invalide = df_clients_dedup.filter(
    F.col("siren").isNotNull() & ~udf_valid_siren(F.col("siren"))
).count()

# Doublons SIREN (même SIREN, silver_client_id différent)
nb_siren_dup = (df_clients_dedup
    .filter(F.col("siren").isNotNull())
    .groupBy("siren").count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"  {'❌' if nb_sans_nom > 0 else '✅'} Sans nom_client        : {nb_sans_nom:,}")
print(f"  {'⚠️' if nb_sans_siren > 0 else '✅'} Sans SIREN             : {nb_sans_siren:,}")
print(f"  {'⚠️' if nb_sans_email > 0 else '✅'} Sans email             : {nb_sans_email:,}")
print(f"  {'⚠️' if nb_sans_cp > 0 else '✅'} Sans code postal       : {nb_sans_cp:,}")
print(f"  {'⚠️' if nb_sans_ville > 0 else '✅'} Sans ville             : {nb_sans_ville:,}")
print(f"  {'⚠️' if nb_email_invalide > 0 else '✅'} Emails format invalide : {nb_email_invalide:,}")
print(f"  {'⚠️' if nb_cp_invalide > 0 else '✅'} CP format invalide     : {nb_cp_invalide:,}")
print(f"  {'⚠️' if nb_siren_invalide > 0 else '✅'} SIREN invalides (Luhn) : {nb_siren_invalide:,}")
print(f"  {'⚠️' if nb_siren_dup > 0 else '✅'} SIREN dupliqués        : {nb_siren_dup:,}")

if nb_siren_invalide > 0:
    print("\n  SIREN invalides (Luhn) :")
    df_clients_dedup.filter(
        F.col("siren").isNotNull() & ~udf_valid_siren(F.col("siren"))
    ).select("nom_client", "siren", "ville_client").show(10, truncate=40)

if nb_siren_dup > 0:
    print("\n  SIREN dupliqués :")
    (df_clients_dedup
        .filter(F.col("siren").isNotNull())
        .groupBy("siren")
        .agg(F.count("silver_client_id").alias("nb"), F.collect_list("nom_client").alias("noms"))
        .filter(F.col("nb") > 1)
        .show(10, truncate=60)
    )

print(f"  ✅ silver_clients écrite ({nb_clients:,} lignes)")


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 8, Finished, Available, Finished, False)


[SILVER] Construction silver_clients...
  Clients dédupliqués : 1,556

  [QUALITÉ CLIENTS]
  ❌ Sans nom_client        : 1,556
  ⚠️ Sans SIREN             : 1,556
  ⚠️ Sans email             : 1,556
  ⚠️ Sans code postal       : 31
  ⚠️ Sans ville             : 149
  ✅ Emails format invalide : 0
  ✅ CP format invalide     : 0
  ✅ SIREN invalides (Luhn) : 0
  ✅ SIREN dupliqués        : 0
  ✅ silver_clients écrite (1,556 lignes)


<a id="cellule-6"></a>

## Cellule 6 — Extraction et normalisation des commerciaux (`silver_commerciaux`)

Construction du **référentiel commerciaux** dédupliqué par nom normalisé.

**Logique de déduplication :**
- La clé est le nom commercial normalisé (trim + consolidation des espaces multiples).
- Même stratégie SHA256 que pour les clients : ID stable et déterministe.

**Enrichissement email :**
- La fonction `generer_email_commercial` reconstruit une adresse `prenom.nom@novaretail.fr` depuis le nom commercial.
- Elle gère les titres de civilité (`M.`, `Mme`, `Mlle`) et supprime les accents via `unicodedata.normalize('NFD')`.
- Cet email généré sert de référence interne ; il n'est pas issu d'une source externe.

**Contrôles qualité :**
- Commerciaux sans fonction renseignée.
- Commerciaux pour lesquels la génération d'email a échoué (nom non décomposable en prénom + nom).

In [7]:
# CELLULE 6 — Extraction et normalisation des COMMERCIAUX
# -----------------------------------------------------------------------------

print("\n[SILVER] Construction silver_commerciaux...")

df_commerciaux_raw = (df_bronze_headers_new
    .filter(F.col("commercial_nom_raw").isNotNull())
    .select(
        F.col("commercial_nom_raw"),
        F.col("commercial_fonction_raw"),
        F.col("ingestion_timestamp"),
        F.col("bronze_id"),
    )
    .withColumn("nom_normalise",
        F.trim(F.regexp_replace(F.col("commercial_nom_raw"), r'\s+', ' ')))
    .withColumn("silver_commercial_id", udf_id(F.col("nom_normalise")))
)

def generer_email_commercial(nom_commercial):
    """Génère prenom.nom@novaretail.fr depuis 'Mme Claire MORIN'"""
    import unicodedata, re
    if nom_commercial is None:
        return None
    nom = re.sub(r'^(?:Mme|M\.|Mr\.?|Mlle)\s+', '', str(nom_commercial).strip(), flags=re.IGNORECASE)
    parties = nom.strip().split()
    if len(parties) >= 2:
        def sans_accents(s):
            return ''.join(c for c in unicodedata.normalize('NFD', s)
                           if unicodedata.category(c) != 'Mn')
        prenom = sans_accents(parties[0].lower())
        nom_f  = sans_accents(parties[-1].lower())
        return f"{prenom}.{nom_f}@novaretail.fr"
    return None

udf_email_com = F.udf(generer_email_commercial, StringType())

df_nb_com = (df_commerciaux_raw
    .groupBy("silver_commercial_id")
    .agg(F.count("bronze_id").alias("nb_commandes"))
)

w_com = Window.partitionBy("silver_commercial_id").orderBy("ingestion_timestamp")
df_commerciaux_dedup = (df_commerciaux_raw
    .withColumn("rn", F.row_number().over(w_com))
    .filter(F.col("rn") == 1)
    .join(df_nb_com, on="silver_commercial_id", how="left")
    .select(
        F.col("silver_commercial_id"),
        F.col("nom_normalise").alias("nom_commercial"),
        F.trim(F.col("commercial_fonction_raw")).alias("fonction_commercial"),
        udf_email_com(F.col("nom_normalise")).alias("email"),
        F.col("nb_commandes"),
        F.lit(SILVER_TIMESTAMP).alias("created_at"),
        F.lit(SILVER_TIMESTAMP).alias("updated_at"),
    )
)

nb_commerciaux = df_commerciaux_dedup.count()
print(f"  Commerciaux dédupliqués : {nb_commerciaux:,}")

(df_commerciaux_dedup.write
    .format("delta").mode("overwrite")
    .option("mergeSchema", "true")
    .save(SILVER_COMMERCIAUX))

# ── Contrôles qualité commerciaux ─────────────────────────────────────────────
print("\n  [QUALITÉ COMMERCIAUX]")
nb_sans_fonction = df_commerciaux_dedup.filter(F.col("fonction_commercial").isNull()).count()
nb_sans_email_c  = df_commerciaux_dedup.filter(F.col("email").isNull()).count()

print(f"  {'⚠️' if nb_sans_fonction > 0 else '✅'} Sans fonction          : {nb_sans_fonction:,}")
print(f"  {'⚠️' if nb_sans_email_c > 0 else '✅'}  Sans email généré      : {nb_sans_email_c:,}")
print("\n  Liste des commerciaux :")
df_commerciaux_dedup.select(
    "nom_commercial", "fonction_commercial", "email", "nb_commandes"
).orderBy(F.desc("nb_commandes")).show(30, truncate=50)
print(f"  ✅ silver_commerciaux écrite ({nb_commerciaux:,} lignes)")


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 9, Finished, Available, Finished, False)


[SILVER] Construction silver_commerciaux...
  Commerciaux dédupliqués : 5

  [QUALITÉ COMMERCIAUX]
  ✅ Sans fonction          : 0
  ✅  Sans email généré      : 0

  Liste des commerciaux :
+-------------------+-----------------------------+------------------------------+------------+
|     nom_commercial|          fonction_commercial|                         email|nb_commandes|
+-------------------+-----------------------------+------------------------------+------------+
|  M. Julien BERNARD|             Chargé de compte|  julien.bernard@novaretail.fr|        4933|
|M. Romain CHEVALIER|         Directeur Commercial|romain.chevalier@novaretail.fr|        4490|
| Mme Sophie LEFRANC|            Directrice Achats|  sophie.lefranc@novaretail.fr|        4287|
|    Mme Laura FAURE|            Chargée de compte|     laura.faure@novaretail.fr|        4168|
|   Mme Claire MORIN|Responsable Approvisionnement|    claire.morin@novaretail.fr|        4045|
+-------------------+---------------------

<a id="cellule-7"></a>

## Cellule 7 — Extraction et normalisation des articles (`silver_articles`)

Construction du **référentiel articles** depuis les lignes Bronze, dédupliqué par référence normalisée.

**Normalisation :**
- La référence est mise en majuscules (`F-001`, `L-042`…) — clé de déduplication et de jointure.
- La déduplication conserve la **désignation la plus récente** (tri `DESC` sur `ingestion_timestamp`), au cas où la désignation évolue dans le temps.

**Enrichissements par UDF :**
- `categorie` : `FRUIT` si référence commence par `F-`, `LEGUME` si `L-`, sinon analyse lexicale de la désignation sur des listes de mots-clés.
- `origine_pays` : extraction regex du pays en fin de désignation (pattern `– Pays`).
- `calibre` : extraction des mentions de calibre/grade si présentes dans la désignation.

**Contrôles qualité :**
- Taux de catégories `INCONNU` (à réduire en enrichissant les règles de catégorisation).
- Références associées à plusieurs désignations distinctes (incohérence catalogue à investiguer).
- Taux de remplissage : origine, désignation, unité de vente.


In [8]:
# CELLULE 7 — Extraction et normalisation des ARTICLES
# -----------------------------------------------------------------------------

print("\n[SILVER] Construction silver_articles...")

df_articles_raw = (df_bronze_lignes_new
    .filter(F.col("reference_article_raw").isNotNull())
    .select(
        F.col("reference_article_raw"),
        F.col("designation_raw"),
        F.col("unite_raw"),
        F.col("ingestion_timestamp"),
    )
    .withColumn("reference_norm",  F.trim(F.upper(F.col("reference_article_raw"))))
    .withColumn("designation_norm", F.trim(F.col("designation_raw")))
    .withColumn("calibre",          udf_calibre(F.col("designation_norm")))
    .withColumn("categorie",        udf_categorie(F.col("reference_norm"), F.col("designation_norm")))
    .withColumn("origine_pays",     udf_origine(F.col("designation_norm")))
    .withColumn("silver_article_id", udf_id(F.col("reference_norm")))
)

w_art = Window.partitionBy("silver_article_id").orderBy(F.desc("ingestion_timestamp"))
df_articles_dedup = (df_articles_raw
    .withColumn("rn", F.row_number().over(w_art))
    .filter(F.col("rn") == 1)
    .select(
        F.col("silver_article_id"),
        F.col("reference_norm").alias("reference_article"),
        F.col("designation_norm").alias("designation"),
        F.col("calibre"),
        F.col("categorie"),
        F.col("origine_pays"),
        F.col("unite_raw").alias("unite_vente"),
        F.lit(SILVER_TIMESTAMP).alias("created_at"),
        F.lit(SILVER_TIMESTAMP).alias("updated_at"),
    )
)

nb_articles = df_articles_dedup.count()

(df_articles_dedup.write
    .format("delta").mode("overwrite")
    .option("mergeSchema", "true")
    .save(SILVER_ARTICLES))

# ── Contrôles qualité articles ────────────────────────────────────────────────
print("\n  [QUALITÉ ARTICLES]")
nb_inconnu     = df_articles_dedup.filter(F.col("categorie") == "INCONNU").count()
nb_sans_origin = df_articles_dedup.filter(F.col("origine_pays").isNull()).count()
nb_sans_desig  = df_articles_dedup.filter(F.col("designation").isNull()).count()
nb_sans_unite  = df_articles_dedup.filter(F.col("unite_vente").isNull()).count()

# Références avec plusieurs désignations différentes (cohérence catalogue)
df_ref_multi_desig = (df_articles_raw
    .groupBy("reference_norm")
    .agg(F.countDistinct("designation_norm").alias("nb_designations"))
    .filter(F.col("nb_designations") > 1)
)
nb_ref_multi = df_ref_multi_desig.count()

print(f"  Articles dédupliqués   : {nb_articles:,}")
print(f"  {'⚠️' if nb_inconnu > 0 else '✅'} Catégorie INCONNU      : {nb_inconnu:,} "
      f"({round(nb_inconnu/nb_articles*100,1) if nb_articles else 0}%)")
print(f"  {'⚠️' if nb_sans_origin > 0 else '✅'} Sans origine_pays      : {nb_sans_origin:,}")
print(f"  {'⚠️' if nb_sans_desig > 0 else '✅'} Sans désignation       : {nb_sans_desig:,}")
print(f"  {'⚠️' if nb_sans_unite > 0 else '✅'} Sans unité vente       : {nb_sans_unite:,}")
print(f"  {'⚠️' if nb_ref_multi > 0 else '✅'} Réf. multi-désignation : {nb_ref_multi:,}")

print("\n  Répartition catégories :")
df_articles_dedup.groupBy("categorie").count().orderBy(F.desc("count")).show()

print("  Top origines pays :")
df_articles_dedup.filter(F.col("origine_pays").isNotNull()).groupBy("origine_pays").count().orderBy(F.desc("count")).show(10)

if nb_inconnu > 0:
    print("  Exemples articles INCONNU (pour améliorer les règles de catégorisation) :")
    df_articles_dedup.filter(F.col("categorie") == "INCONNU").select(
        "reference_article", "designation", "unite_vente"
    ).show(10, truncate=60)

if nb_ref_multi > 0:
    print("  Références avec désignations multiples :")
    (df_ref_multi_desig.join(
        df_articles_raw.select("reference_norm", "designation_norm").distinct(),
        on="reference_norm", how="left"
    ).orderBy("reference_norm").show(20, truncate=60))

print(f"  ✅ silver_articles écrite ({nb_articles:,} lignes)")


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 10, Finished, Available, Finished, False)


[SILVER] Construction silver_articles...

  [QUALITÉ ARTICLES]
  Articles dédupliqués   : 289
  ⚠️ Catégorie INCONNU      : 54 (18.7%)
  ⚠️ Sans origine_pays      : 248
  ✅ Sans désignation       : 0
  ✅ Sans unité vente       : 0
  ⚠️ Réf. multi-désignation : 280

  Répartition catégories :
+---------+-----+
|categorie|count|
+---------+-----+
|   LEGUME|  129|
|    FRUIT|  106|
|  INCONNU|   54|
+---------+-----+

  Top origines pays :
+------------+-----+
|origine_pays|count|
+------------+-----+
|        fins|   19|
|      France|    7|
|     Espagne|    6|
|      Italie|    3|
|       Maroc|    2|
|       Pérou|    1|
|     Turquie|    1|
|       Kenya|    1|
|    Équateur|    1|
+------------+-----+

  Exemples articles INCONNU (pour améliorer les règles de catégorisation) :
+-----------------+------------------+--------------+
|reference_article|       designation|   unite_vente|
+-----------------+------------------+--------------+
|    CA-2026-79846| Asperges blanches|   Bott

<a id="cellule-8"></a>

## Cellule 8 — Construction des commandes Silver (`silver_commandes`)

Transformation et validation des headers Bronze en commandes Silver, avec jointure vers les référentiels clients et commerciaux.

**Étapes clés :**

1. **Typage des champs bruts** : montants via `udf_montant`, dates via `udf_date`, TVA en `double` après suppression du symbole `%`.

2. **Fallback numéro de commande** : si `numero_commande_raw` est absent du contenu du document, le numéro est extrait par regex depuis le nom du fichier source (`BC-2026-53738.docx → BC-2026-53738`). Normalisation finale : underscores → tirets, mise en majuscules.

3. **Résolution des FK** : jointure gauche vers `silver_clients` et `silver_commerciaux` sur les IDs SHA256 recalculés à la volée — pas de clé technique transmise depuis Bronze.

4. **Dédoublonnage intra-run** : `ROW_NUMBER()` sur `numero_commande` conserve la version la plus récente ; les doublons reçoivent le statut `DOUBLON`.

5. **Validation étendue** :
   - `OK` : numéro commande présent, date parsée, au moins un montant disponible.
   - `INCOMPLET` : champ obligatoire manquant.
   - `DOUBLON` : numéro de commande en double dans le run.

6. **Écriture Delta partitionnée** par `periode_zip` pour optimiser les lectures filtrées en Gold.

**Contrôles post-écriture :**
- FK non résolues (clients/commerciaux sans correspondance Silver).
- Dates aberrantes (futures, antérieures à 2020, livraison avant commande).
- TVA hors barème légal français (0%, 2,1%, 5,5%, 10%, 20%).
- Incohérence HT / TTC / TVA (écart > 1€).
- Champs facultatifs à faible taux de remplissage (mode règlement, incoterm…).


In [9]:
# CELLULE 8 — Construction des COMMANDES Silver
# -----------------------------------------------------------------------------

print("\n[SILVER] Construction silver_commandes...")

df_silver_clients     = spark.read.format("delta").load(SILVER_CLIENTS)
df_silver_commerciaux = spark.read.format("delta").load(SILVER_COMMERCIAUX)

df_commandes_typed = (df_bronze_headers_new
    .withColumn("total_ht_float",  udf_montant(F.col("total_ht_raw")))
    .withColumn("total_ttc_float", udf_montant(F.col("total_ttc_raw")))
    .withColumn("tva_float",
        F.when(F.col("tva_raw").isNotNull(),
               F.regexp_replace(F.col("tva_raw"), r'[%\s]', '').cast("double"))
    )
    .withColumn("date_commande_iso",  udf_date(F.col("date_commande_raw")))
    .withColumn("date_livraison_iso", udf_date(F.col("date_livraison_souhaitee_raw")))
    # Fallback numéro commande depuis nom_fichier_docx si absent du contenu
    # Pattern : BC-2026-53738.docx → BC-2026-53738
    .withColumn("numero_commande",
        F.when(
            F.col("numero_commande_raw").isNotNull() & (F.trim(F.col("numero_commande_raw")) != ""),
            F.trim(F.col("numero_commande_raw"))
        ).otherwise(
            F.regexp_extract(
                F.col("nom_fichier_docx"),
                r'((?:BC|BDC|CMD|BON)[_\-]\d{4}[_\-]\d+)',
                1
            )
        )
    )
    .withColumn("numero_commande",
        # Normaliser les underscores en tirets + vider string vide → NULL
        F.when(
            F.col("numero_commande").isNotNull() & (F.col("numero_commande") != ""),
            F.upper(F.regexp_replace(F.col("numero_commande"), "_", "-"))
        ).otherwise(F.lit(None).cast(StringType()))
    )
    .withColumn("lieu_livraison",     F.trim(F.col("lieu_livraison_raw")))
    .withColumn("cle_client",
        F.lower(F.regexp_replace(F.coalesce(F.col("lieu_livraison_raw"), F.lit("")), r'\s+', ' ')))
    .withColumn("silver_client_id_calc", udf_id(F.col("cle_client")))
    .withColumn("com_norm",
        F.trim(F.regexp_replace(F.coalesce(F.col("commercial_nom_raw"), F.lit("")), r'\s+', ' ')))
    .withColumn("silver_commercial_id_calc",
        F.when(F.col("com_norm") != "", udf_id(F.col("com_norm"))))
)

df_commandes_joined = (df_commandes_typed
    .join(df_silver_clients.select("silver_client_id"),
          df_commandes_typed["silver_client_id_calc"] == df_silver_clients["silver_client_id"], "left")
    .join(df_silver_commerciaux.select("silver_commercial_id"),
          df_commandes_typed["silver_commercial_id_calc"] == df_silver_commerciaux["silver_commercial_id"], "left")
)

w_dedup = Window.partitionBy("numero_commande").orderBy(F.desc("ingestion_timestamp"))
df_commandes_dedup = (df_commandes_joined
    .withColumn("rn_dedup", F.row_number().over(w_dedup))
    .withColumn("is_doublon", F.when(F.col("rn_dedup") > 1, True).otherwise(False))
)

# ── Validation étendue ────────────────────────────────────────────────────────
DATE_MIN = F.lit("2020-01-01").cast(DateType())
DATE_MAX = F.lit("2030-12-31").cast(DateType())
TVA_VALIDES = [0.0, 2.1, 5.5, 10.0, 20.0]

df_commandes_final = (df_commandes_dedup
    .withColumn("silver_commande_id", udf_id(F.col("bronze_id")))

    # Date commande parsée pour les contrôles
    .withColumn("_date_commande_parsed", F.to_date(F.col("date_commande_iso"), "yyyy-MM-dd"))
    .withColumn("_date_livraison_parsed", F.to_date(F.col("date_livraison_iso"), "yyyy-MM-dd"))

    # ── Statut validation ──────────────────────────────────────────────────
    .withColumn("statut_validation",
        F.when(F.col("is_doublon"), F.lit("DOUBLON"))
        .when(F.col("numero_commande").isNull() | F.col("date_commande_iso").isNull(), F.lit("INCOMPLET"))
        .when(F.col("total_ht_float").isNull() & F.col("total_ttc_float").isNull(), F.lit("INCOMPLET"))
        .otherwise(F.lit("OK"))
    )
    .withColumn("motif_rejet",
        F.when(F.col("is_doublon"),
               F.concat(F.lit("Doublon BDC "), F.col("numero_commande")))
        .when(F.col("numero_commande").isNull(), F.lit("N° commande manquant"))
        .when(F.col("date_commande_iso").isNull(), F.lit("Date commande invalide"))
        .when(F.col("total_ht_float").isNull() & F.col("total_ttc_float").isNull(),
              F.lit("Montants HT et TTC manquants"))
        .otherwise(F.lit(None).cast(StringType()))
    )
    .select(
        F.col("silver_commande_id"),
        F.col("bronze_id"),
        F.col("silver_client_id").alias("silver_client_id"),
        F.col("silver_commercial_id").alias("silver_commercial_id"),
        F.col("numero_commande"),
        F.to_date(F.col("date_commande_iso"), "yyyy-MM-dd").alias("date_commande"),
        F.to_date(F.col("date_livraison_iso"), "yyyy-MM-dd").alias("date_livraison_souhaitee"),
        F.col("heure_livraison_souhaitee_raw").alias("heure_livraison_souhaitee"),
        F.col("lieu_livraison"),
        F.trim(F.col("ref_contrat_raw")).alias("ref_contrat"),
        F.trim(F.col("mode_livraison_raw")).alias("mode_livraison"),
        F.trim(F.col("mode_reglement_raw")).alias("mode_reglement"),
        F.trim(F.col("incoterm_raw")).alias("incoterm"),
        F.col("total_ht_float").cast(DecimalType(12,2)).alias("total_ht"),
        F.col("total_ttc_float").cast(DecimalType(12,2)).alias("total_ttc"),
        F.col("tva_float").cast(DecimalType(5,2)).alias("tva_taux"),
        F.coalesce(F.col("devise_raw"), F.lit("EUR")).alias("devise"),
        F.col("fournisseur_raw").alias("fournisseur"),
        F.col("nb_lignes_articles").alias("nb_lignes"),
        F.col("statut_validation"),
        F.col("motif_rejet"),
        F.col("periode_zip"),
        F.col("format_source"),
        F.col("nom_fichier_docx").alias("source_fichier"),
        F.col("ingestion_timestamp"),
        F.lit(SILVER_TIMESTAMP).alias("silver_timestamp"),
    )
    .orderBy("date_commande", "numero_commande")
)

nb_ok       = df_commandes_final.filter(F.col("statut_validation") == "OK").count()
nb_incomplet= df_commandes_final.filter(F.col("statut_validation") == "INCOMPLET").count()
nb_doublon  = df_commandes_final.filter(F.col("statut_validation") == "DOUBLON").count()
nb_total    = nb_ok + nb_incomplet + nb_doublon

print(f"  Commandes total    : {nb_total:,}")
print(f"  ✅ OK              : {nb_ok:,}")
print(f"  ⚠️  Incomplet      : {nb_incomplet:,}")
print(f"  🔄 Doublon         : {nb_doublon:,}")

(df_commandes_final.write
    .format("delta").mode("overwrite")
    .option("mergeSchema", "true")
    .partitionBy("periode_zip")
    .save(SILVER_COMMANDES))

# ── Contrôles qualité commandes ────────────────────────────────────────────────
print("\n  [QUALITÉ COMMANDES]")
df_com_check = spark.read.format("delta").load(SILVER_COMMANDES)
df_com_ok    = df_com_check.filter(F.col("statut_validation") == "OK")

# FK
nb_fk_cli_ko = df_com_ok.filter(F.col("silver_client_id").isNull()).count()
nb_fk_com_ko = df_com_ok.filter(F.col("silver_commercial_id").isNull()).count()
print(f"  {'⚠️' if nb_fk_cli_ko > 0 else '✅'} FK client non résolue  : {nb_fk_cli_ko:,}")
print(f"  {'⚠️' if nb_fk_com_ko > 0 else '✅'} FK commercial non rés. : {nb_fk_com_ko:,}")

# Dates aberrantes
nb_date_future = df_com_ok.filter(
    F.col("date_commande") > F.current_date()
).count()
nb_date_ancienne = df_com_ok.filter(
    F.col("date_commande") < F.lit("2020-01-01").cast(DateType())
).count()
nb_livraison_avant_commande = df_com_ok.filter(
    F.col("date_livraison_souhaitee").isNotNull() &
    (F.col("date_livraison_souhaitee") < F.col("date_commande"))
).count()
print(f"  {'⚠️' if nb_date_future > 0 else '✅'} Dates commande futures : {nb_date_future:,}")
print(f"  {'⚠️' if nb_date_ancienne > 0 else '✅'} Dates commande < 2020 : {nb_date_ancienne:,}")
print(f"  {'⚠️' if nb_livraison_avant_commande > 0 else '✅'} Livraison < commande   : {nb_livraison_avant_commande:,}")

# TVA aberrante
nb_tva_invalide = df_com_ok.filter(
    F.col("tva_taux").isNotNull() &
    ~F.col("tva_taux").isin([float(t) for t in [0, 2.1, 5.5, 10.0, 20.0]])
).count()
print(f"  {'⚠️' if nb_tva_invalide > 0 else '✅'} TVA hors barème légal  : {nb_tva_invalide:,}")

# Cohérence HT vs TTC via TVA
nb_coherence_ht_ttc = df_com_ok.filter(
    F.col("total_ht").isNotNull() &
    F.col("total_ttc").isNotNull() &
    F.col("tva_taux").isNotNull() &
    (F.abs(
        F.col("total_ttc").cast("double") -
        F.round(F.col("total_ht").cast("double") * (1 + F.col("tva_taux").cast("double") / 100), 2)
    ) > 1.0)
).count()
print(f"  {'⚠️' if nb_coherence_ht_ttc > 0 else '✅'} Incoh. HT/TTC/TVA     : {nb_coherence_ht_ttc:,}")

# Montants suspects (nuls ou négatifs)
nb_ht_nul     = df_com_ok.filter(F.col("total_ht").isNotNull() & (F.col("total_ht") <= 0)).count()
nb_ht_giant   = df_com_ok.filter(F.col("total_ht").isNotNull() & (F.col("total_ht") > 500000)).count()
print(f"  {'⚠️' if nb_ht_nul > 0 else '✅'} Montant HT ≤ 0         : {nb_ht_nul:,}")
print(f"  {'⚠️' if nb_ht_giant > 0 else '✅'} Montant HT > 500k€     : {nb_ht_giant:,}")

# Champs facultatifs vides
for champ in ["mode_reglement", "incoterm", "ref_contrat", "mode_livraison"]:
    nb_vide = df_com_ok.filter(F.col(champ).isNull()).count()
    pct = round(nb_vide / nb_ok * 100, 1) if nb_ok > 0 else 0
    ico = "⚠️" if pct > 30 else "✅"
    print(f"  {ico} {champ:25s} manquant : {nb_vide:,} ({pct}%)")

# Cohérence nb_lignes header vs lignes bronze
nb_header_zero_lignes = df_com_ok.filter(
    F.col("nb_lignes").isNotNull() & (F.col("nb_lignes") == 0)
).count()
print(f"  {'⚠️' if nb_header_zero_lignes > 0 else '✅'} Headers avec 0 lignes  : {nb_header_zero_lignes:,}")

if nb_date_future > 0 or nb_livraison_avant_commande > 0:
    print("\n  Commandes avec dates suspectes :")
    df_com_ok.filter(
        (F.col("date_commande") > F.current_date()) |
        (F.col("date_livraison_souhaitee").isNotNull() &
         (F.col("date_livraison_souhaitee") < F.col("date_commande")))
    ).select("numero_commande", "date_commande", "date_livraison_souhaitee",
             "source_fichier").show(10, truncate=50)

if nb_tva_invalide > 0:
    print("\n  TVA hors barème légal [0, 2.1, 5.5, 10, 20]% :")
    df_com_ok.filter(
        F.col("tva_taux").isNotNull() &
        ~F.col("tva_taux").isin([float(t) for t in [0, 2.1, 5.5, 10.0, 20.0]])
    ).select("numero_commande", "tva_taux", "total_ht", "total_ttc").show(10)

print(f"\n  ✅ silver_commandes écrite ({nb_total:,} lignes)")


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 11, Finished, Available, Finished, False)


[SILVER] Construction silver_commandes...
  Commandes total    : 22,306
  ✅ OK              : 9,999
  ⚠️  Incomplet      : 383
  🔄 Doublon         : 11,924

  [QUALITÉ COMMANDES]
  ✅ FK client non résolue  : 0
  ✅ FK commercial non rés. : 0
  ✅ Dates commande futures : 0
  ✅ Dates commande < 2020 : 0
  ✅ Livraison < commande   : 0
  ✅ TVA hors barème légal  : 0
  ✅ Incoh. HT/TTC/TVA     : 0
  ✅ Montant HT ≤ 0         : 0
  ✅ Montant HT > 500k€     : 0
  ⚠️ mode_reglement            manquant : 4,993 (49.9%)
  ⚠️ incoterm                  manquant : 4,993 (49.9%)
  ⚠️ ref_contrat               manquant : 6,258 (62.6%)
  ⚠️ mode_livraison            manquant : 4,993 (49.9%)
  ✅ Headers avec 0 lignes  : 0

  ✅ silver_commandes écrite (22,306 lignes)


<a id="cellule-9"></a>

## Cellule 9 — Construction des lignes commande Silver (`silver_lignes_commande`)

Transformation des lignes Bronze en lignes Silver typées, avec calcul du montant théorique et détection des anomalies de valorisation.

**Enrichissements par ligne :**

| Champ calculé | Formule |
|---------------|---------|
| `montant_ht_calcule` | `quantite × prix_unitaire_ht × (1 − remise_pct / 100)` |
| `ecart_montant` | `|montant_ht_ligne − montant_ht_calcule|` |
| `tva_taux` | Issu de la ligne si renseigné, sinon hérité du header commande |

**Statut de la ligne :**
- `OK` : ligne complète et montant cohérent (écart ≤ 0,05€).
- `ANOMALIE_MONTANT` : écart de calcul > 0,05€ — signale une remise mal appliquée ou une saisie erronée.
- `INCOMPLET` : désignation et quantité toutes deux nulles.

**Résolution FK article :**
Jointure gauche vers `silver_articles` sur la référence normalisée. Les lignes sans correspondance (référence inconnue du catalogue) restent dans Silver avec `silver_article_id = NULL`.

**Contrôles qualité post-écriture :**
- Prix unitaires et quantités nuls ou négatifs.
- Quantités anormalement élevées (> 10 000 unités).
- Remises supérieures à 100% (erreur de saisie) ou > 50% (suspect).
- Cohérence entre `nb_lignes` déclaré dans le header et le nombre réel de lignes traitées.


In [10]:
# CELLULE 9 — Construction des LIGNES COMMANDE Silver
# -----------------------------------------------------------------------------

print("\n[SILVER] Construction silver_lignes_commande...")

df_silver_commandes = spark.read.format("delta").load(SILVER_COMMANDES)
df_silver_articles  = spark.read.format("delta").load(SILVER_ARTICLES)

df_mapping     = df_silver_commandes.select("silver_commande_id", "bronze_id", "tva_taux").distinct()
df_mapping_art = df_silver_articles.select("silver_article_id", F.col("reference_article").alias("ref_norm"))

df_lignes_typed = (df_bronze_lignes_new
    .join(df_mapping, on="bronze_id", how="inner")
    .withColumn("quantite_float",   udf_montant(F.col("quantite_raw")))
    .withColumn("prix_ht_float",    udf_montant(F.col("prix_unitaire_raw")))
    .withColumn("montant_ht_float", udf_montant(F.col("montant_ht_raw")))
    .withColumn("remise_float",
        F.when(F.col("remise_raw").isNotNull(), udf_montant(F.col("remise_raw"))).otherwise(F.lit(0.0)))
    .withColumn("tva_ligne_float",
        F.when(F.col("tva_raw").isNotNull(),
               F.regexp_replace(F.col("tva_raw"), r'[%\s]', '').cast("double"))
        .otherwise(F.col("tva_taux").cast("double")))
    .withColumn("date_liv_iso", udf_date(F.col("date_livraison_raw")))
    .withColumn("ref_norm_ligne",
        F.when(F.col("reference_article_raw").isNotNull(),
               F.trim(F.upper(F.col("reference_article_raw")))))
    .join(df_mapping_art, F.col("ref_norm_ligne") == F.col("ref_norm"), "left")
    .withColumn("montant_calcule",
        F.when(F.col("quantite_float").isNotNull() & F.col("prix_ht_float").isNotNull(),
               F.round(F.col("quantite_float") * F.col("prix_ht_float") *
                       (1 - F.coalesce(F.col("remise_float"), F.lit(0.0)) / 100), 2))
    )
    .withColumn("ecart_montant",
        F.when(F.col("montant_ht_float").isNotNull() & F.col("montant_calcule").isNotNull(),
               F.round(F.abs(F.col("montant_ht_float") - F.col("montant_calcule")), 2))
    )
    .withColumn("silver_ligne_id", udf_id(F.col("bronze_ligne_id")))
    .withColumn("statut_ligne",
        F.when(F.col("designation_raw").isNull() & F.col("quantite_float").isNull(),
               F.lit("INCOMPLET"))
        .when(F.col("ecart_montant").isNotNull() & (F.col("ecart_montant") > 0.05),
               F.lit("ANOMALIE_MONTANT"))
        .otherwise(F.lit("OK"))
    )
    .select(
        F.col("silver_ligne_id"),
        F.col("silver_commande_id"),
        F.col("silver_article_id"),
        F.col("bronze_ligne_id"),
        F.col("index_ligne"),
        F.col("ref_norm_ligne").alias("reference_article"),
        F.trim(F.col("designation_raw")).alias("designation"),
        F.col("unite_raw").alias("calibre"),
        F.col("quantite_float").cast(DecimalType(10,3)).alias("quantite"),
        F.col("unite_raw").alias("unite"),
        F.col("prix_ht_float").cast(DecimalType(10,4)).alias("prix_unitaire_ht"),
        F.col("remise_float").cast(DecimalType(5,2)).alias("remise_pct"),
        F.col("montant_ht_float").cast(DecimalType(12,2)).alias("montant_ht_ligne"),
        F.col("montant_calcule").cast(DecimalType(12,2)).alias("montant_ht_calcule"),
        F.col("ecart_montant").cast(DecimalType(12,2)).alias("ecart_montant"),
        F.col("tva_ligne_float").cast(DecimalType(5,2)).alias("tva_taux"),
        F.to_date(F.col("date_liv_iso"), "yyyy-MM-dd").alias("date_livraison_ligne"),
        F.col("statut_ligne"),
        F.lit(SILVER_TIMESTAMP).alias("silver_timestamp"),
    )
)

nb_lignes_ok       = df_lignes_typed.filter(F.col("statut_ligne") == "OK").count()
nb_lignes_anomalie = df_lignes_typed.filter(F.col("statut_ligne") == "ANOMALIE_MONTANT").count()
nb_lignes_incompl  = df_lignes_typed.filter(F.col("statut_ligne") == "INCOMPLET").count()
nb_lignes_total    = nb_lignes_ok + nb_lignes_anomalie + nb_lignes_incompl

print(f"  Lignes total          : {nb_lignes_total:,}")
print(f"  ✅ OK                 : {nb_lignes_ok:,}")
print(f"  ⚠️  Anomalie montant  : {nb_lignes_anomalie:,}")
print(f"  ❌ Incomplet          : {nb_lignes_incompl:,}")

(df_lignes_typed.write
    .format("delta").mode("overwrite")
    .option("mergeSchema", "true")
    .save(SILVER_LIGNES_COMMANDE))

# ── Contrôles qualité lignes ──────────────────────────────────────────────────
print("\n  [QUALITÉ LIGNES]")

# Prix négatifs ou nuls
nb_prix_nul = df_lignes_typed.filter(
    F.col("prix_unitaire_ht").isNotNull() & (F.col("prix_unitaire_ht") <= 0)
).count()
# Quantités aberrantes
nb_qte_neg = df_lignes_typed.filter(
    F.col("quantite").isNotNull() & (F.col("quantite") <= 0)
).count()
nb_qte_giant = df_lignes_typed.filter(
    F.col("quantite").isNotNull() & (F.col("quantite") > 10000)
).count()
# Remise > 100%
nb_remise_invalide = df_lignes_typed.filter(
    F.col("remise_pct").isNotNull() & (F.col("remise_pct") > 100)
).count()
# Remise > 50% (suspect mais pas impossible)
nb_remise_forte = df_lignes_typed.filter(
    F.col("remise_pct").isNotNull() & (F.col("remise_pct") > 50) & (F.col("remise_pct") <= 100)
).count()
# Articles non référencés (pas de silver_article_id)
nb_sans_ref_art = df_lignes_typed.filter(F.col("silver_article_id").isNull()).count()
# Sans référence article du tout
nb_sans_ref     = df_lignes_typed.filter(F.col("reference_article").isNull()).count()
# Sans désignation
nb_sans_desig_l = df_lignes_typed.filter(F.col("designation").isNull()).count()

print(f"  {'⚠️' if nb_prix_nul > 0 else '✅'} Prix unitaire ≤ 0       : {nb_prix_nul:,}")
print(f"  {'⚠️' if nb_qte_neg > 0 else '✅'} Quantité ≤ 0            : {nb_qte_neg:,}")
print(f"  {'⚠️' if nb_qte_giant > 0 else '✅'} Quantité > 10 000       : {nb_qte_giant:,}")
print(f"  {'❌' if nb_remise_invalide > 0 else '✅'} Remise > 100%           : {nb_remise_invalide:,}")
print(f"  {'⚠️' if nb_remise_forte > 0 else '✅'} Remise > 50%            : {nb_remise_forte:,}")
print(f"  {'⚠️' if nb_sans_ref_art > 0 else '✅'} Réf. hors référentiel  : {nb_sans_ref_art:,} "
      f"({round(nb_sans_ref_art/nb_lignes_total*100,1) if nb_lignes_total else 0}%)")
print(f"  {'⚠️' if nb_sans_ref > 0 else '✅'} Sans référence article  : {nb_sans_ref:,}")
print(f"  {'⚠️' if nb_sans_desig_l > 0 else '✅'} Sans désignation        : {nb_sans_desig_l:,}")

# Cohérence nb_lignes header vs lignes réelles
print("\n  Cohérence nb_lignes header vs lignes réelles :")
df_silver_com_check = spark.read.format("delta").load(SILVER_COMMANDES).filter(
    F.col("statut_validation") == "OK"
)
df_nb_lignes_reelles = (df_lignes_typed
    .groupBy("silver_commande_id")
    .agg(F.count("silver_ligne_id").alias("nb_lignes_reelles"))
)
df_coherence_nb = (df_silver_com_check
    .select("silver_commande_id", "numero_commande", "nb_lignes", "periode_zip")
    .join(df_nb_lignes_reelles, on="silver_commande_id", how="left")
    .withColumn("ecart_nb",
        F.abs(F.col("nb_lignes").cast("int") - F.coalesce(F.col("nb_lignes_reelles"), F.lit(0))))
    .filter(F.col("ecart_nb") > 0)
)
nb_ecart_lignes = df_coherence_nb.count()
print(f"  {'⚠️' if nb_ecart_lignes > 0 else '✅'} Headers nb_lignes ≠ réel : {nb_ecart_lignes:,}")
if nb_ecart_lignes > 0:
    df_coherence_nb.select(
        "numero_commande", "nb_lignes", "nb_lignes_reelles", "ecart_nb", "periode_zip"
    ).orderBy(F.desc("ecart_nb")).show(10)

if nb_remise_invalide > 0:
    print("\n  Lignes avec remise > 100% :")
    df_lignes_typed.filter(F.col("remise_pct") > 100).select(
        "reference_article", "designation", "quantite", "prix_unitaire_ht",
        "remise_pct", "montant_ht_ligne"
    ).show(10, truncate=40)

print(f"  ✅ silver_lignes_commande écrite ({nb_lignes_total:,} lignes)")


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 12, Finished, Available, Finished, False)


[SILVER] Construction silver_lignes_commande...
  Lignes total          : 193,901
  ✅ OK                 : 192,332
  ⚠️  Anomalie montant  : 1,569
  ❌ Incomplet          : 0

  [QUALITÉ LIGNES]
  ✅ Prix unitaire ≤ 0       : 0
  ✅ Quantité ≤ 0            : 0
  ✅ Quantité > 10 000       : 0
  ✅ Remise > 100%           : 0
  ✅ Remise > 50%            : 0
  ✅ Réf. hors référentiel  : 0 (0.0%)
  ✅ Sans référence article  : 0
  ✅ Sans désignation        : 0

  Cohérence nb_lignes header vs lignes réelles :
  ⚠️ Headers nb_lignes ≠ réel : 5,006
+---------------+---------+-----------------+--------+-----------+
|numero_commande|nb_lignes|nb_lignes_reelles|ecart_nb|periode_zip|
+---------------+---------+-----------------+--------+-----------+
|  BC-2024-00199|       13|               78|      65|    2024-04|
|  BC-2024-00243|       13|               78|      65|    2024-04|
|  BC-2024-00321|       13|               78|      65|    2024-04|
|  BC-2024-00781|       13|               78|      65

<a id="cellule-10"></a>

## Cellule 10 — Enregistrement de l'audit Silver

Écriture d'une ligne de traçabilité dans la table `silver_audit` en mode **append** (conservation de l'historique des runs).

**Métriques enregistrées :**
- Compteurs Bronze en entrée (`nb_headers_bronze`, `nb_lignes_bronze`).
- Compteurs Silver produits par entité (clients, commerciaux, articles).
- Répartition des statuts commandes et lignes.
- Durée d'exécution en secondes (delta entre `SILVER_TIMESTAMP` et `datetime.now()` à la fin du run).

Cette table est la base du **monitoring opérationnel** du pipeline : elle permet de détecter des dérives de qualité ou des anomalies de volumétrie entre deux runs sans relire les tables métier.

In [11]:
# CELLULE 10 — Audit Silver
# -----------------------------------------------------------------------------

print("\n[SILVER] Enregistrement audit...")

fin_silver = datetime.now(timezone.utc)
duree_s    = int((fin_silver - SILVER_TIMESTAMP).total_seconds())

df_audit = spark.createDataFrame(
    [{
        "run_id":                 SILVER_RUN_ID,
        "silver_timestamp":       SILVER_TIMESTAMP,
        "nb_headers_bronze":      int(nb_headers),
        "nb_lignes_bronze":       int(nb_lignes),
        "nb_clients_crees":       int(nb_clients),
        "nb_commerciaux_crees":   int(nb_commerciaux),
        "nb_articles_crees":      int(nb_articles),
        "nb_commandes_ok":        int(nb_ok),
        "nb_commandes_incomplet": int(nb_incomplet),
        "nb_commandes_doublon":   int(nb_doublon),
        "nb_lignes_ok":           int(nb_lignes_ok),
        "nb_lignes_anomalie":     int(nb_lignes_anomalie + nb_lignes_incompl),
        "duree_secondes":         duree_s,
    }],
    schema=SCHEMA_SILVER_AUDIT
)
df_audit.write.format("delta").mode("append").save(SILVER_AUDIT)

StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 13, Finished, Available, Finished, False)


[SILVER] Enregistrement audit...


<a id="cellule-11"></a>

## Cellule 11 — Vérifications post-Silver et analyses exploratoires

Production du **rapport d'exécution résumé** et de quatre analyses de valeur exploitables directement après ingestion.

**Rapport :** synthèse des compteurs clés (référentiels, statuts commandes et lignes).

**Analyses produites :**

1. **CA mensuel par période et format source** — permet de détecter si un format de fichier génère des montants aberrants ou un volume anormal.

2. **Top 10 clients par CA HT** — premier aperçu de la concentration client, réutilisable en Gold pour un scoring RFM.

3. **Top 10 articles par CA HT** — permet d'identifier les références à fort enjeu pour les contrôles de prix et de stock.

4. **Répartition par catégorie article** — valide la cohérence de la catégorisation FRUIT / LEGUME et détecte les INCONNU résiduels.

5. **Distribution des délais commande → livraison** — statistiques descriptives (moyenne, médiane, min, max) par période, utiles pour l'analyse logistique.


In [12]:
# CELLULE 11 — Vérifications post-Silver
# -----------------------------------------------------------------------------

print("\n" + "="*65)
print("  RAPPORT SILVER")
print("="*65)
print(f"  Durée              : {duree_s}s")
print(f"  Clients            : {nb_clients:,}")
print(f"  Commerciaux        : {nb_commerciaux:,}")
print(f"  Articles           : {nb_articles:,}")
print(f"  Commandes OK       : {nb_ok:,}")
print(f"  Commandes incompl. : {nb_incomplet:,}")
print(f"  Commandes doublons : {nb_doublon:,}")
print(f"  Lignes OK          : {nb_lignes_ok:,}")
print(f"  Lignes anomalies   : {nb_lignes_anomalie:,}")
print("="*65)

# ── Répartition par période et format ─────────────────────────────────────────
print("\n── CA mensuel par période et format source ──")
(spark.read.format("delta").load(SILVER_COMMANDES)
    .filter(F.col("statut_validation") == "OK")
    .groupBy("periode_zip", "format_source")
    .agg(
        F.count("silver_commande_id").alias("nb_commandes"),
        F.round(F.sum("total_ht"), 2).alias("ca_ht_total"),
        F.round(F.avg("total_ht"), 2).alias("panier_moyen_ht"),
    )
    .orderBy("periode_zip", "format_source")
    .show(100))

# ── Top 10 clients par CA ──────────────────────────────────────────────────────
print("\n── Top 10 clients par CA HT ──")
(spark.read.format("delta").load(SILVER_COMMANDES)
    .filter(F.col("statut_validation") == "OK")
    .join(spark.read.format("delta").load(SILVER_CLIENTS).select(
        "silver_client_id", "nom_client", "ville_client"),
        on="silver_client_id", how="left")
    .groupBy("silver_client_id", "nom_client", "ville_client")
    .agg(
        F.count("silver_commande_id").alias("nb_commandes"),
        F.round(F.sum("total_ht"), 2).alias("ca_ht_total"),
        F.round(F.avg("total_ht"), 2).alias("panier_moyen"),
    )
    .orderBy(F.desc("ca_ht_total"))
    .show(10, truncate=50))

# ── Top 10 articles par CA ─────────────────────────────────────────────────────
print("\n── Top 10 articles par CA HT ──")
(spark.read.format("delta").load(SILVER_LIGNES_COMMANDE)
    .filter(F.col("statut_ligne") == "OK")
    .groupBy("reference_article", "designation")
    .agg(
        F.round(F.sum("montant_ht_ligne"), 2).alias("ca_ht_total"),
        F.sum("quantite").alias("qte_totale"),
        F.round(F.avg("prix_unitaire_ht"), 4).alias("prix_moyen"),
        F.count("silver_ligne_id").alias("nb_commandes"),
    )
    .orderBy(F.desc("ca_ht_total"))
    .show(10, truncate=50))

# ── Répartition par catégorie article ─────────────────────────────────────────
print("\n── CA par catégorie article ──")
(spark.read.format("delta").load(SILVER_LIGNES_COMMANDE)
    .filter(F.col("statut_ligne") == "OK")
    .join(spark.read.format("delta").load(SILVER_ARTICLES).select(
        "silver_article_id", "categorie"), on="silver_article_id", how="left")
    .groupBy("categorie")
    .agg(
        F.count("silver_ligne_id").alias("nb_lignes"),
        F.round(F.sum("montant_ht_ligne"), 2).alias("ca_ht_total"),
        F.round(F.avg("montant_ht_ligne"), 2).alias("montant_moyen"),
    )
    .orderBy(F.desc("ca_ht_total"))
    .show())

# ── Distribution des délais de livraison ──────────────────────────────────────
print("\n── Distribution des délais commande → livraison (jours) ──")
(spark.read.format("delta").load(SILVER_COMMANDES)
    .filter(F.col("statut_validation") == "OK")
    .filter(F.col("date_livraison_souhaitee").isNotNull())
    .withColumn("delai_jours", F.datediff(F.col("date_livraison_souhaitee"), F.col("date_commande")))
    .groupBy("periode_zip")
    .agg(
        F.round(F.avg("delai_jours"), 1).alias("delai_moyen"),
        F.min("delai_jours").alias("delai_min"),
        F.max("delai_jours").alias("delai_max"),
        F.round(F.percentile_approx("delai_jours", 0.5), 1).alias("delai_median"),
    )
    .orderBy("periode_zip")
    .show(50))


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 14, Finished, Available, Finished, False)


  RAPPORT SILVER
  Durée              : 433s
  Clients            : 1,556
  Commerciaux        : 5
  Articles           : 289
  Commandes OK       : 9,999
  Commandes incompl. : 383
  Commandes doublons : 11,924
  Lignes OK          : 192,332
  Lignes anomalies   : 1,569

── CA mensuel par période et format source ──
+-----------+-------------+------------+-----------+---------------+
|periode_zip|format_source|nb_commandes|ca_ht_total|panier_moyen_ht|
+-----------+-------------+------------+-----------+---------------+
|    2024-04|         DOCX|         422| 2816334.38|        6673.78|
|    2024-05|         DOCX|         442| 2830246.54|        6403.27|
|    2024-06|         DOCX|         384| 2443264.51|        6362.67|
|    2024-07|         DOCX|         446| 2890329.13|        6480.56|
|    2024-08|         DOCX|         418| 2688675.03|        6432.24|
|    2024-09|         DOCX|         399| 2615513.67|        6555.17|
|    2024-10|         DOCX|         452| 2970321.16|       

<a id="cellule-12"></a>

## Cellule 12 — Diagnostic des anomalies de montant

Analyse approfondie des lignes classifiées `ANOMALIE_MONTANT` pour orienter les actions correctives.

**Si des anomalies sont présentes, trois analyses sont produites :**

1. **Distribution statistique des écarts** (min, moyenne, médiane, max, cumul) : permet de distinguer des écarts systématiques (erreur de règle de calcul) d'écarts sporadiques (saisie ponctuelle).

2. **Top des anomalies par montant d'écart** : lignes individuelles triées par écart décroissant, avec détail prix/quantité/remise pour identifier la source de l'erreur.

3. **Articles les plus touchés** : agrégation par référence article pour détecter si une famille de produits est structurellement mal valorisée (ex. remise codée en dur au mauvais endroit).

4. **Commandes les plus impactées** : calcul de l'écart cumulé et du ratio `ecart / total_ht` par commande — prioritise les dossiers à retraiter en urgence.

**Lignes INCOMPLET :** affichage des lignes avec désignation et quantité toutes deux nulles, pour investigation en amont (qualité des fichiers sources).


In [13]:
# CELLULE 12 — Diagnostic des anomalies de montant
# -----------------------------------------------------------------------------

print("\n[DIAGNOSTIC] Analyse des anomalies de montant...")

df_lignes_silver = spark.read.format("delta").load(SILVER_LIGNES_COMMANDE)
df_anomalies     = df_lignes_silver.filter(F.col("statut_ligne") == "ANOMALIE_MONTANT")
nb_anomalies     = df_anomalies.count()
print(f"  Total anomalies montant : {nb_anomalies:,}")

if nb_anomalies > 0:
    # Distribution statistique des écarts
    print("\n── Statistiques écarts (€) ──")
    df_anomalies.agg(
        F.round(F.min("ecart_montant"), 2).alias("ecart_min"),
        F.round(F.avg("ecart_montant"), 2).alias("ecart_moyen"),
        F.round(F.percentile_approx("ecart_montant", 0.5), 2).alias("ecart_median"),
        F.round(F.max("ecart_montant"), 2).alias("ecart_max"),
        F.round(F.sum("ecart_montant"), 2).alias("ecart_cumule"),
    ).show()

    # Top anomalies
    print("\n── Top anomalies (écart le plus élevé) ──")
    df_anomalies.select(
        F.round(F.col("ecart_montant"), 2).alias("ecart_eur"),
        "reference_article", "designation", "quantite",
        "prix_unitaire_ht", "remise_pct", "montant_ht_ligne", "montant_ht_calcule"
    ).orderBy(F.desc("ecart_eur")).show(20, truncate=50)

    # Articles les plus touchés
    print("\n── Articles les plus touchés ──")
    (df_anomalies
        .groupBy("reference_article", "designation")
        .agg(
            F.count("silver_ligne_id").alias("nb_anomalies"),
            F.round(F.sum("ecart_montant"), 2).alias("ecart_cumule"),
            F.round(F.avg("ecart_montant"), 2).alias("ecart_moyen"),
        )
        .orderBy(F.desc("ecart_cumule"))
        .show(15, truncate=50))

    # Anomalies par commande — jointure Spark (pas de collect)
    print("\n── Commandes les plus impactées ──")
    df_com = spark.read.format("delta").load(SILVER_COMMANDES)
    (df_anomalies
        .join(df_com.select("silver_commande_id", "numero_commande", "date_commande",
                            "periode_zip", "total_ht"),
              on="silver_commande_id", how="left")
        .groupBy("numero_commande", "date_commande", "periode_zip", "total_ht")
        .agg(
            F.count("silver_ligne_id").alias("nb_lignes_anomalie"),
            F.round(F.sum("ecart_montant"), 2).alias("ecart_total"),
            F.round(F.sum("ecart_montant") / F.col("total_ht").cast("double") * 100, 2)
             .alias("ecart_pct_ca"),
        )
        .orderBy(F.desc("ecart_total"))
        .show(20, truncate=50))
else:
    print("  ✅ Aucune anomalie de montant détectée")

# ── Lignes INCOMPLET ──────────────────────────────────────────────────────────
df_incomplet_l = df_lignes_silver.filter(F.col("statut_ligne") == "INCOMPLET")
nb_incomplet_l = df_incomplet_l.count()
print(f"\n[DIAGNOSTIC] Lignes INCOMPLET : {nb_incomplet_l:,}")
if nb_incomplet_l > 0:
    df_incomplet_l.select(
        "silver_ligne_id", "reference_article", "designation",
        "quantite", "prix_unitaire_ht", "statut_ligne"
    ).show(10, truncate=50)


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 15, Finished, Available, Finished, False)


[DIAGNOSTIC] Analyse des anomalies de montant...
  Total anomalies montant : 1,569

── Statistiques écarts (€) ──
+---------+-----------+------------+---------+------------+
|ecart_min|ecart_moyen|ecart_median|ecart_max|ecart_cumule|
+---------+-----------+------------+---------+------------+
|     0.76|      45.64|       26.96|   668.45|    71603.12|
+---------+-----------+------------+---------+------------+


── Top anomalies (écart le plus élevé) ──
+---------+-----------------+------------------+--------+----------------+----------+----------------+------------------+
|ecart_eur|reference_article|       designation|quantite|prix_unitaire_ht|remise_pct|montant_ht_ligne|montant_ht_calcule|
+---------+-----------------+------------------+--------+----------------+----------+----------------+------------------+
|   668.45|    CA-2026-43898|        Framboises| 792.000|         16.8800|      5.00|        13368.96|          12700.51|
|   501.63|    CA-2026-99341|        Framboises| 739.

<a id="cellule-13"></a>

## Cellule 13 — Contrôles de cohérence inter-tables Silver

Série de **5 contrôles d'intégrité croisée** entre les tables Silver pour détecter des incohérences non visibles dans les contrôles intra-table.

| # | Contrôle | Seuil d'alerte |
|---|----------|----------------|
| 1 | `total_ht` commande vs somme des `montant_ht_ligne` | Écart > 1€ |
| 2 | Commandes OK sans aucune ligne Silver | > 0 |
| 3 | Lignes orphelines (sans commande parente) | > 0 |
| 4 | Réconciliation CA mensuel commandes vs lignes | Delta en % par période |
| 5 | Intégrité FK `lignes → commandes` et `lignes → articles` | > 0 |

**Point technique :** les anti-jointures (contrôles 3 et 5) sont réalisées en Spark (`left_anti`) sans `collect()` pour éviter de ramener les données en mémoire driver — essentiel pour la scalabilité sur de grands volumes.

Le contrôle 4 (réconciliation CA) est le plus stratégique : un delta significatif entre le CA déclaré dans les headers et le CA reconstitué depuis les lignes indique soit des lignes manquantes, soit des montants de header non fiables.


In [14]:
# CELLULE 13 — Contrôles de cohérence inter-tables Silver
# -----------------------------------------------------------------------------

print("\n[COHÉRENCE] Contrôles inter-tables Silver...\n")

df_com = spark.read.format("delta").load(SILVER_COMMANDES).filter(F.col("statut_validation") == "OK")
df_lig = spark.read.format("delta").load(SILVER_LIGNES_COMMANDE).filter(F.col("statut_ligne") == "OK")

# ── 1. Cohérence total_ht commande vs somme lignes ────────────────────────────
print("── 1. Cohérence total_ht (commande vs somme lignes) ──")
df_somme_lignes = (df_lig
    .groupBy("silver_commande_id")
    .agg(F.round(F.sum("montant_ht_ligne"), 2).alias("somme_ht_lignes"))
)
df_ecart_com = (df_com
    .select("silver_commande_id", "numero_commande", "date_commande",
            "total_ht", "nb_lignes", "periode_zip")
    .join(df_somme_lignes, on="silver_commande_id", how="left")
    .withColumn("ecart_ht",
        F.round(F.abs(F.col("total_ht").cast("double") - F.coalesce(F.col("somme_ht_lignes"), F.lit(0.0))), 2))
    .withColumn("alerte",
        F.when(F.col("ecart_ht") > 1.0, F.lit("⚠️ ECART > 1€"))
        .when(F.col("somme_ht_lignes").isNull(), F.lit("❌ SANS LIGNES"))
        .otherwise(F.lit("✅ OK")))
)
nb_ecart    = df_ecart_com.filter(F.col("ecart_ht") > 1.0).count()
nb_sans_lig = df_ecart_com.filter(F.col("somme_ht_lignes").isNull()).count()
nb_ok_coh   = df_ecart_com.filter(F.col("alerte") == "✅ OK").count()
print(f"  ✅ Commandes cohérentes   : {nb_ok_coh:,}")
print(f"  ⚠️  Écart > 1€            : {nb_ecart:,}")
print(f"  ❌ Commandes sans lignes  : {nb_sans_lig:,}")
if nb_ecart > 0:
    df_ecart_com.filter(F.col("ecart_ht") > 1.0).select(
        "numero_commande", "date_commande", "periode_zip",
        "total_ht", "somme_ht_lignes", "ecart_ht"
    ).orderBy(F.desc("ecart_ht")).show(20, truncate=50)

# ── 2. Commandes sans lignes ───────────────────────────────────────────────────
print("\n── 2. Commandes OK sans lignes Silver ──")
if nb_sans_lig > 0:
    df_ecart_com.filter(F.col("somme_ht_lignes").isNull()).select(
        "numero_commande", "date_commande", "periode_zip", "nb_lignes", "total_ht"
    ).show(20, truncate=50)
else:
    print("  ✅ Toutes les commandes OK ont au moins une ligne")

# ── 3. Lignes orphelines — via anti-join Spark (pas de collect) ───────────────
print("\n── 3. Lignes orphelines (sans commande parente) ──")
df_all_lig = spark.read.format("delta").load(SILVER_LIGNES_COMMANDE)
df_orphelin = df_all_lig.join(
    df_com.select("silver_commande_id"),
    on="silver_commande_id", how="left_anti"
)
nb_orphelin = df_orphelin.count()
print(f"  {'❌' if nb_orphelin > 0 else '✅'} Lignes orphelines : {nb_orphelin:,}")
if nb_orphelin > 0:
    df_orphelin.select(
        "silver_ligne_id", "silver_commande_id", "reference_article", "montant_ht_ligne"
    ).show(10, truncate=50)

# ── 4. Réconciliation CA mensuel ───────────────────────────────────────────────
print("\n── 4. Réconciliation CA mensuel (commandes vs lignes) ──")
df_ca_com = (df_com.groupBy("periode_zip").agg(
    F.count("silver_commande_id").alias("nb_commandes"),
    F.round(F.sum("total_ht"), 2).alias("ca_ht_commandes"),
))
df_ca_lig = (df_lig
    .join(df_com.select("silver_commande_id", "periode_zip"), on="silver_commande_id", how="left")
    .groupBy("periode_zip")
    .agg(F.round(F.sum("montant_ht_ligne"), 2).alias("ca_ht_lignes"))
)
(df_ca_com.join(df_ca_lig, on="periode_zip", how="outer")
    .withColumn("delta_ca", F.round(
        F.coalesce(F.col("ca_ht_commandes"), F.lit(0.0)) -
        F.coalesce(F.col("ca_ht_lignes"), F.lit(0.0)), 2))
    .withColumn("delta_pct", F.when(F.col("ca_ht_commandes").isNotNull() & (F.col("ca_ht_commandes") != 0),
        F.round(F.col("delta_ca") / F.col("ca_ht_commandes") * 100, 2)))
    .orderBy("periode_zip")
    .show(50))

# ── 5. Intégrité référentielle FK ──────────────────────────────────────────────
print("\n── 5. Intégrité référentielle FK lignes → commandes, articles ──")
# Lignes avec silver_commande_id absent de silver_commandes (tous statuts)
df_all_com = spark.read.format("delta").load(SILVER_COMMANDES)
df_lig_all = spark.read.format("delta").load(SILVER_LIGNES_COMMANDE)
nb_fk_com_lig = df_lig_all.join(
    df_all_com.select("silver_commande_id"),
    on="silver_commande_id", how="left_anti"
).count()
nb_fk_art_lig = df_lig_all.filter(
    F.col("silver_article_id").isNull() & F.col("reference_article").isNotNull()
).count()
print(f"  {'❌' if nb_fk_com_lig > 0 else '✅'} Lignes sans commande parente : {nb_fk_com_lig:,}")
print(f"  {'⚠️' if nb_fk_art_lig > 0 else '✅'} Lignes réf non cataloguée   : {nb_fk_art_lig:,}")

print("\n[COHÉRENCE] Terminé")


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 16, Finished, Available, Finished, False)


[COHÉRENCE] Contrôles inter-tables Silver...

── 1. Cohérence total_ht (commande vs somme lignes) ──
  ✅ Commandes cohérentes   : 1,075
  ⚠️  Écart > 1€            : 8,924
  ❌ Commandes sans lignes  : 0
+---------------+-------------+-----------+--------+---------------+--------+
|numero_commande|date_commande|periode_zip|total_ht|somme_ht_lignes|ecart_ht|
+---------------+-------------+-----------+--------+---------------+--------+
|  BC-2024-00222|   2024-04-16|    2024-04|17547.26|      105283.56| 87736.3|
|  BC-2024-00455|   2024-05-02|    2024-05|16464.86|      101844.48|85379.62|
|  BC-2024-00061|   2024-04-03|    2024-04|16365.09|       99685.80|83320.71|
|  BC-2024-00772|   2024-05-27|    2024-05|15746.67|       95918.82|80172.15|
|  BC-2024-00187|   2024-04-12|    2024-04|15121.86|       93537.30|78415.44|
|  BC-2024-00451|   2024-05-02|    2024-05|14830.16|       90796.92|75966.76|
|  BC-2024-00847|   2024-05-30|    2024-05|15081.96|       90491.76| 75409.8|
|  BC-2024-00398

<a id="cellule-14"></a>

## Cellule 14 — Rapport final Silver, score qualité global et nettoyage

Clôture du notebook avec trois actions finales.

**1. Nettoyage des fichiers temporaires**  
Suppression des fichiers `novaretail_*` dans `/tmp` pour libérer de l'espace sur le nœud driver entre les runs.

**2. Score qualité global**  
Calcul d'un indicateur synthétique (moyenne de 5 métriques) exprimé sur 100 :
- Taux de commandes OK
- Taux de lignes OK
- Taux de clients avec nom
- Taux de clients avec SIREN
- Taux d'articles catégorisés (hors INCONNU)

Grille de lecture : 🟢 ≥ 95 | 🟡 ≥ 80 | 🟠 ≥ 60 | 🔴 < 60.

**3. Rapport final formaté**  
Affichage consolidé de tous les compteurs clés du run avec indication vers l'étape suivante : **NB_03 Gold Aggregation**.

In [15]:
# CELLULE 14 — Rapport final Silver + score qualité global + nettoyage
# -----------------------------------------------------------------------------

import shutil
from pathlib import Path

# ── Nettoyage /tmp ─────────────────────────────────────────────────────────────
tmp_dir = Path("/tmp")
for _f in tmp_dir.glob("novaretail_*"):
    try:
        _f.unlink() if _f.is_file() else shutil.rmtree(_f)
    except Exception:
        pass
print("[CLEAN] Fichiers temporaires /tmp nettoyés")

# ── Score qualité global ────────────────────────────────────────────────────────
# Calcul d'un score synthétique basé sur les métriques clés
fin_silver   = datetime.now(timezone.utc)
duree_totale = int((fin_silver - SILVER_TIMESTAMP).total_seconds())

_metriques = {
    "taux_commandes_ok":      round(nb_ok / max(nb_total, 1) * 100, 1),
    "taux_lignes_ok":         round(nb_lignes_ok / max(nb_lignes_ok + nb_lignes_anomalie + nb_lignes_incompl, 1) * 100, 1),
    "taux_clients_avec_nom":  round((nb_clients - nb_sans_nom) / max(nb_clients, 1) * 100, 1),
    "taux_clients_avec_siren":round((nb_clients - nb_sans_siren) / max(nb_clients, 1) * 100, 1),
    "taux_articles_categorises": round((nb_articles - nb_inconnu) / max(nb_articles, 1) * 100, 1),
}
_score = round(sum(_metriques.values()) / len(_metriques), 1)

print(f"\n{'='*65}")
print(f"  RAPPORT FINAL SILVER — {SILVER_RUN_ID}")
print(f"{'='*65}")
print(f"  Durée totale        : {duree_totale}s")
print(f"  Nouveaux headers    : {nb_new:,}")
print(f"{'─'*65}")
print(f"  Référentiels :")
print(f"    Clients           : {nb_clients:,}")
print(f"    Commerciaux       : {nb_commerciaux:,}")
print(f"    Articles          : {nb_articles:,}")
print(f"{'─'*65}")
print(f"  Commandes :")
print(f"    ✅ OK             : {nb_ok:,} ({_metriques['taux_commandes_ok']}%)")
print(f"    ⚠️  Incomplet     : {nb_incomplet:,}")
print(f"    🔄 Doublon        : {nb_doublon:,}")
print(f"{'─'*65}")
print(f"  Lignes articles :")
print(f"    ✅ OK             : {nb_lignes_ok:,} ({_metriques['taux_lignes_ok']}%)")
print(f"    ⚠️  Anomalie HT   : {nb_lignes_anomalie:,}")
print(f"    ❌ Incomplet      : {nb_lignes_incompl:,}")
print(f"{'─'*65}")
print(f"  Score qualité global : {_score:.1f} / 100")
_grade = "🟢 EXCELLENT" if _score >= 95 else ("🟡 BON" if _score >= 80 else ("🟠 MOYEN" if _score >= 60 else "🔴 INSUFFISANT"))
print(f"  Niveau               : {_grade}")
print(f"{'─'*65}")
for k, v in _metriques.items():
    ico = "✅" if v >= 95 else ("⚠️" if v >= 80 else "❌")
    print(f"  {ico} {k:35s} : {v:.1f}%")
print(f"{'='*65}")
print(f"\n  → Prochaine étape : NB_03 Gold Aggregation")


StatementMeta(, ad77449e-aa05-4c6c-ae7c-9d57c2b24b4d, 17, Finished, Available, Finished, False)

[CLEAN] Fichiers temporaires /tmp nettoyés

  RAPPORT FINAL SILVER — SILVER_20260608_124354
  Durée totale        : 459s
  Nouveaux headers    : 22,306
─────────────────────────────────────────────────────────────────
  Référentiels :
    Clients           : 1,556
    Commerciaux       : 5
    Articles          : 289
─────────────────────────────────────────────────────────────────
  Commandes :
    ✅ OK             : 9,999 (44.8%)
    ⚠️  Incomplet     : 383
    🔄 Doublon        : 11,924
─────────────────────────────────────────────────────────────────
  Lignes articles :
    ✅ OK             : 192,332 (99.2%)
    ⚠️  Anomalie HT   : 1,569
    ❌ Incomplet      : 0
─────────────────────────────────────────────────────────────────
  Score qualité global : 45.1 / 100
  Niveau               : 🔴 INSUFFISANT
─────────────────────────────────────────────────────────────────
  ❌ taux_commandes_ok                   : 44.8%
  ✅ taux_lignes_ok                      : 99.2%
  ❌ taux_clients_avec_n